In [ ]:
# ============================================================
# RSNA Knee MRI - Soft Label Production
# Step 1: Locate and verify saved inputs
# ============================================================

from pathlib import Path
import pandas as pd
import json
import os

OUTPUT_DIR = Path("/kaggle/working/llm_soft_label_production")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Competition data
# ------------------------------------------------------------

TRAIN_FILE = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection/train.csv"
)

assert TRAIN_FILE.exists(), f"Missing competition file: {TRAIN_FILE}"

# ------------------------------------------------------------
# Previous notebook output
# Search only expected folders—do not scan the MRI dataset
# ------------------------------------------------------------

demo_files = (
    list(
        Path("/kaggle/input/notebooks").glob(
            "*/*/llm_pseudo_labels_v2/"
            "soft_label_v1_production_demonstrations.csv"
        )
    )
    +
    list(
        Path("/kaggle/input").glob(
            "*/llm_pseudo_labels_v2/"
            "soft_label_v1_production_demonstrations.csv"
        )
    )
)

assert len(demo_files) == 1, (
    "Expected exactly one production-demonstrations file.\n"
    f"Files found: {demo_files}"
)

DEMO_FILE = demo_files[0]

print("Competition file:", TRAIN_FILE)
print("Demonstration file:", DEMO_FILE)

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

train = pd.read_csv(TRAIN_FILE)
production_demos = pd.read_csv(DEMO_FILE)

TARGET_COLS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# ------------------------------------------------------------
# Separate gold-labelled and unlabelled studies
# ------------------------------------------------------------

gold_mask = train[TARGET_COLS].notna().all(axis=1)

gold_df = (
    train.loc[gold_mask]
    .copy()
    .reset_index(drop=True)
)

unlabelled_df = (
    train.loc[~gold_mask]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(train) == 4407, f"Unexpected train rows: {len(train)}"
assert len(gold_df) == 58, f"Unexpected gold rows: {len(gold_df)}"
assert len(unlabelled_df) == 4349, (
    f"Unexpected unlabelled rows: {len(unlabelled_df)}"
)
assert train["Report"].isna().sum() == 0, "Some reports are missing."
assert len(production_demos) == 10, (
    f"Expected 10 demonstrations; found {len(production_demos)}"
)

print("\nTrain rows:", len(train))
print("Gold studies:", len(gold_df))
print("Unlabelled studies:", len(unlabelled_df))
print("Production demonstrations:", len(production_demos))
print("Output directory:", OUTPUT_DIR)

print("\nStep 1 input verification passed ✓")

In [ ]:
# ============================================================
# Step 2: Verify API secret and demonstration structure
# ============================================================

from kaggle_secrets import UserSecretsClient

# Install/import the current OpenAI Python package
!pip install -q -U openai

from openai import OpenAI
import openai

# Read the key securely without displaying it
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("rsna-knee-soft-label-production")

assert api_key is not None, "rsna-knee-soft-label-production was not found."
assert len(api_key.strip()) > 20, "rsna-knee-soft-label-productionappears invalid."

client = OpenAI(api_key=api_key)

print("OpenAI package version:", openai.__version__)
print("API secret loaded securely ✓")
print("No API request has been made ✓")

print("\nDemonstration file columns:")
print(production_demos.columns.tolist())

print("\nDemonstration shape:")
print(production_demos.shape)

print("\nDemonstration data types:")
print(production_demos.dtypes)

print("\nStep 2 setup verification passed ✓")

In [ ]:
# ============================================================
# Step 3: Build the Soft Label V1 production prompt
# ============================================================

from pydantic import BaseModel, Field
from typing import List
import json

MODEL_NAME = "gpt-5.6-terra"

# ------------------------------------------------------------
# Structured API response format
# ------------------------------------------------------------

class TargetPrediction(BaseModel):
    target: str
    score: float = Field(ge=0.0, le=1.0)
    confidence: float = Field(ge=0.0, le=1.0)
    evidence: str


class ReportPrediction(BaseModel):
    study_uid: str
    predictions: List[TargetPrediction]


# ------------------------------------------------------------
# Convert the 10 gold demonstrations into compact text
# ------------------------------------------------------------

def make_demonstration(row):
    labels = {
        target: int(row[target])
        for target in TARGET_COLS
    }

    return (
        f"REFERENCE REPORT:\n{row['Report']}\n\n"
        f"OFFICIAL DATASET LABELS:\n"
        f"{json.dumps(labels, ensure_ascii=False)}"
    )


PRODUCTION_EXAMPLES_TEXT = "\n\n---\n\n".join(
    make_demonstration(row)
    for _, row in production_demos.iterrows()
)

# ------------------------------------------------------------
# Instructions kept constant to improve prompt caching
# ------------------------------------------------------------

SYSTEM_PROMPT = """
You label knee MRI radiology reports for a dataset with a
dataset-specific annotation convention.

Return continuous soft scores, not hard pseudo-labels.

For every target:
- score is the probability that the dataset label is positive.
- confidence measures confidence in your interpretation of the report
  and the dataset convention.
- evidence must be short and grounded in the report.
- read the entire report, including findings and conclusion.
- handle reports written in any language.
- distinguish explicit absence from a finding that is merely unmentioned.
- use the gold reference examples to infer the dataset's labeling convention.
- do not infer Synovitis solely from the presence of effusion.
- do not infer Baker's cyst unless a popliteal/Baker cyst is supported.
- distinguish focal cartilage injury from osteoarthritis when possible.

Return exactly one prediction for each of these targets, in this order:
ACL
MCL
Medial Meniscus
Lateral Meniscus
Medial OA
Lateral OA
PF OA
Effusion
Synovitis
Baker's
Contusion
Fracture
""".strip()


def build_production_prompt(study_uid, report):
    return f"""
The following are gold-labelled reference examples from the same dataset.

{PRODUCTION_EXAMPLES_TEXT}

Now label this new report.

STUDY UID:
{study_uid}

NEW REPORT:
{report}

Return all 12 target predictions using continuous scores from 0 to 1.
The study_uid in your response must exactly match the supplied STUDY UID.
""".strip()


# ------------------------------------------------------------
# Local validation only — no API request
# ------------------------------------------------------------

test_row = unlabelled_df.iloc[0]

test_prompt = build_production_prompt(
    str(test_row["StudyInstanceUID"]),
    str(test_row["Report"])
)

assert len(production_demos) == 10
assert all(target in production_demos.columns for target in TARGET_COLS)
assert str(test_row["StudyInstanceUID"]) in test_prompt
assert str(test_row["Report"]) in test_prompt

print("Model:", MODEL_NAME)
print("Production demonstrations:", len(production_demos))
print("Test prompt characters:", len(test_prompt))
print("Structured response schema ready ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 4: Restart-safe checkpoint and response validation
# ============================================================

import os
import time
from datetime import datetime, timezone

CHECKPOINT_FILE = OUTPUT_DIR / "production_predictions.jsonl"
ERROR_FILE = OUTPUT_DIR / "production_errors.jsonl"
SNAPSHOT_FILE = OUTPUT_DIR / "production_predictions_snapshot.csv"

# A run will never process more than this number unless we
# deliberately change it.
MAX_REPORTS_PER_RUN = 25


def append_jsonl(path, record):
    """
    Append one completed result and force it to disk immediately.
    """
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        file.flush()
        os.fsync(file.fileno())


def read_jsonl_safely(path):
    """
    Read valid records while safely ignoring a damaged final line.
    """
    records = []

    if not path.exists():
        return records

    with open(path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(
                    f"Warning: ignored damaged JSONL line "
                    f"{line_number} in {path.name}"
                )

    return records


def validate_prediction(parsed, expected_uid):
    """
    Reject incomplete, duplicated or misaligned API responses.
    """
    assert parsed.study_uid == expected_uid, (
        "Returned study UID does not match requested UID."
    )

    predictions = parsed.predictions

    assert len(predictions) == len(TARGET_COLS), (
        f"Expected 12 predictions; received {len(predictions)}."
    )

    returned_targets = [
        prediction.target for prediction in predictions
    ]

    assert returned_targets == TARGET_COLS, (
        "Targets are missing, duplicated or in the wrong order.\n"
        f"Returned: {returned_targets}"
    )

    for prediction in predictions:
        assert 0.0 <= prediction.score <= 1.0
        assert 0.0 <= prediction.confidence <= 1.0
        assert prediction.evidence.strip()

    return True


def call_label_api(study_uid, report):
    """
    Make exactly one API request.
    This function is only defined here—it is not called in Step 4.
    """
    response = client.responses.parse(
        model=MODEL_NAME,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": build_production_prompt(
                    study_uid,
                    report,
                ),
            },
        ],
        text_format=ReportPrediction,
    )

    parsed = response.output_parsed

    if parsed is None:
        raise ValueError("API response did not contain parsed output.")

    validate_prediction(parsed, study_uid)

    usage = getattr(response, "usage", None)

    return {
        "study_uid": study_uid,
        "model": MODEL_NAME,
        "response_id": response.id,
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "input_tokens": getattr(usage, "input_tokens", None),
        "output_tokens": getattr(usage, "output_tokens", None),
        "predictions": [
            prediction.model_dump()
            for prediction in parsed.predictions
        ],
    }


def completed_study_uids():
    records = read_jsonl_safely(CHECKPOINT_FILE)

    return {
        str(record["study_uid"])
        for record in records
        if "study_uid" in record
    }


def save_csv_snapshot():
    """
    Convert all successful JSONL checkpoints into a readable CSV.
    """
    records = read_jsonl_safely(CHECKPOINT_FILE)
    rows = []

    for record in records:
        row = {
            "StudyInstanceUID": record["study_uid"],
            "Model": record.get("model"),
            "ResponseID": record.get("response_id"),
            "CreatedAtUTC": record.get("created_at_utc"),
            "InputTokens": record.get("input_tokens"),
            "OutputTokens": record.get("output_tokens"),
        }

        for prediction in record["predictions"]:
            target = prediction["target"]
            row[f"{target}_score"] = prediction["score"]
            row[f"{target}_confidence"] = prediction["confidence"]
            row[f"{target}_evidence"] = prediction["evidence"]

        rows.append(row)

    snapshot = pd.DataFrame(rows)
    snapshot.to_csv(SNAPSHOT_FILE, index=False)

    return snapshot


already_completed = completed_study_uids()

print("Checkpoint file:", CHECKPOINT_FILE)
print("Already completed:", len(already_completed))
print("Maximum reports per run:", MAX_REPORTS_PER_RUN)
print("Sequential processing only ✓")
print("Immediate disk checkpointing ready ✓")
print("Step 4 made no API requests ✓")

In [ ]:
# ============================================================
# RESUME CELL: Restore saved production checkpoint
# Run after Steps 1–4 in a new Kaggle session
# ============================================================

import shutil

saved_checkpoint_candidates = (
    list(Path("/kaggle/input").glob(
        "*/production_predictions.jsonl"
    ))
    +
    list(Path("/kaggle/input").glob(
        "*/llm_soft_label_production/"
        "production_predictions.jsonl"
    ))
    +
    list(Path("/kaggle/input/notebooks").glob(
        "*/*/llm_soft_label_production/"
        "production_predictions.jsonl"
    ))
)

assert saved_checkpoint_candidates, (
    "No saved production checkpoint was found. "
    "Attach the latest notebook output or upload "
    "production_predictions.jsonl as an Input."
)

# The largest checkpoint should contain the most completed reports
SAVED_CHECKPOINT = max(
    saved_checkpoint_candidates,
    key=lambda path: path.stat().st_size,
)

saved_records = read_jsonl_safely(SAVED_CHECKPOINT)

if CHECKPOINT_FILE.exists():
    working_records = read_jsonl_safely(CHECKPOINT_FILE)
else:
    working_records = []

if len(saved_records) > len(working_records):
    shutil.copy2(
        SAVED_CHECKPOINT,
        CHECKPOINT_FILE,
    )

restored_records = read_jsonl_safely(CHECKPOINT_FILE)
restored_uids = {
    str(record["study_uid"])
    for record in restored_records
}

assert len(restored_records) == len(restored_uids)

# Definitions required by the final Luna production function
LUNA_MODEL = "gpt-5.6-luna"

SOFT_LABEL_SYSTEM_PROMPT = """
You are analysing knee MRI radiology reports for a multilabel
medical-imaging dataset.

For each of the 12 targets, estimate the probability that the
dataset's official binary label is 1.

Important:
- Return a continuous score from 0.0 to 1.0.
- Do not convert predictions into hard 0/1 labels.
- Distinguish confirmed abnormalities from explicitly normal findings.
- Account for negation, uncertainty, chronic/postoperative findings,
  synonyms and multilingual terminology.
- Do not assume that an unmentioned finding is definitely negative.
- Use the entire report.
- Confidence describes confidence in extracting the label from the
  report, not the severity of the condition.

Score guidance:
- 0.00–0.10: clearly negative
- 0.10–0.35: probably negative or not clearly supported
- 0.35–0.65: uncertain, ambiguous or insufficiently described
- 0.65–0.90: probably positive
- 0.90–1.00: clearly positive

Confidence guidance:
- 0.90–1.00: explicit and unambiguous evidence
- 0.60–0.89: reasonably supported but indirect or mildly uncertain
- 0.30–0.59: ambiguous, incomplete or difficult to interpret
- 0.00–0.29: insufficient evidence
""".strip()

restored_snapshot = save_csv_snapshot()

print("Saved checkpoint:", SAVED_CHECKPOINT)
print("Restored checkpoint records:", len(restored_records))
print("Unique restored studies:", len(restored_uids))
print("Reports remaining:", 4349 - len(restored_uids))
print("Checkpoint restoration passed ✓")

In [ ]:
# ============================================================
# Step 5: Locate the previous 12-report production pilot
# ============================================================

pilot_files = (
    list(
        Path("/kaggle/input/notebooks").glob(
            "*/*/llm_pseudo_labels_v2/"
            "soft_label_v1_terra_production_pilot.csv"
        )
    )
    +
    list(
        Path("/kaggle/input").glob(
            "*/llm_pseudo_labels_v2/"
            "soft_label_v1_terra_production_pilot.csv"
        )
    )
)

assert len(pilot_files) == 1, (
    "Expected exactly one Terra production pilot file.\n"
    f"Files found: {pilot_files}"
)

PILOT_FILE = pilot_files[0]
pilot_df = pd.read_csv(PILOT_FILE)

print("Pilot file:", PILOT_FILE)
print("Pilot shape:", pilot_df.shape)

print("\nPilot columns:")
for column in pilot_df.columns:
    print(column)

print("\nUnique studies:", pilot_df["StudyInstanceUID"].nunique())

print("\nStep 5 pilot recovery inspection passed ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 6: Import the 12 pilot predictions into the checkpoint
# ============================================================

valid_unlabelled_uids = set(
    unlabelled_df["StudyInstanceUID"].astype(str)
)

already_completed = completed_study_uids()

imported_count = 0
skipped_count = 0

for _, row in pilot_df.iterrows():
    study_uid = str(row["StudyInstanceUID"])

    assert study_uid in valid_unlabelled_uids, (
        f"Pilot UID is not in the unlabelled dataset: {study_uid}"
    )

    if study_uid in already_completed:
        skipped_count += 1
        continue

    predictions = []

    for target in TARGET_COLS:
        score = float(row[f"{target}_score"])
        confidence = float(row[f"{target}_confidence"])
        evidence = str(row[f"{target}_evidence"]).strip()

        assert 0.0 <= score <= 1.0
        assert 0.0 <= confidence <= 1.0
        assert evidence and evidence.lower() != "nan"

        predictions.append(
            {
                "target": target,
                "score": score,
                "confidence": confidence,
                "evidence": evidence,
            }
        )

    record = {
        "study_uid": study_uid,
        "model": str(row["model"]),
        "response_id": None,
        "created_at_utc": None,
        "source": "recovered_previous_terra_pilot",
        "input_tokens": None,
        "output_tokens": None,
        "predictions": predictions,
    }

    append_jsonl(CHECKPOINT_FILE, record)

    already_completed.add(study_uid)
    imported_count += 1

snapshot = save_csv_snapshot()

checkpoint_uids = completed_study_uids()

assert len(checkpoint_uids) == len(snapshot)
assert snapshot["StudyInstanceUID"].nunique() == len(snapshot)
assert len(checkpoint_uids) >= 12

print("Pilot predictions imported:", imported_count)
print("Already present and skipped:", skipped_count)
print("Total checkpointed studies:", len(checkpoint_uids))
print("Snapshot rows:", len(snapshot))
print("Snapshot saved:", SNAPSHOT_FILE)

print("\nThe 12 pilot predictions will not be requested again ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 7: Guarded sequential production runner
# ============================================================

# Disable automatic SDK retries so one notebook iteration cannot
# silently create several paid requests.
client = OpenAI(
    api_key=api_key,
    max_retries=0,
    timeout=180.0,
)


def get_remaining_reports():
    completed = completed_study_uids()

    remaining = unlabelled_df[
        ~unlabelled_df["StudyInstanceUID"]
        .astype(str)
        .isin(completed)
    ].copy()

    return remaining.reset_index(drop=True)


def run_production_chunk(max_reports=1):
    """
    Process a deliberately small number of reports sequentially.

    Safety properties:
    - never exceeds MAX_REPORTS_PER_RUN;
    - never requests an already checkpointed study;
    - writes each success immediately;
    - refresh can lose at most the one request currently in flight;
    - stops after the first error.
    """

    assert isinstance(max_reports, int)
    assert 1 <= max_reports <= MAX_REPORTS_PER_RUN, (
        f"max_reports must be between 1 and "
        f"{MAX_REPORTS_PER_RUN}."
    )

    remaining = get_remaining_reports()
    selected = remaining.head(max_reports)

    print("Already checkpointed:", len(completed_study_uids()))
    print("Reports remaining:", len(remaining))
    print("Reports authorized for this run:", len(selected))
    print("Model:", MODEL_NAME)
    print()

    successful = 0

    for position, row in selected.iterrows():
        study_uid = str(row["StudyInstanceUID"])
        report = str(row["Report"])

        # Final duplicate protection immediately before the call
        if study_uid in completed_study_uids():
            print("Skipped completed study:", study_uid)
            continue

        print(
            f"Request {position + 1}/{len(selected)} | "
            f"Study: {study_uid}"
        )

        try:
            record = call_label_api(
                study_uid=study_uid,
                report=report,
            )

            # Save before starting another request
            append_jsonl(CHECKPOINT_FILE, record)
            save_csv_snapshot()

            successful += 1

            print("  Saved immediately ✓")

        except Exception as error:
            error_record = {
                "study_uid": study_uid,
                "model": MODEL_NAME,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "error_type": type(error).__name__,
                "error_message": str(error),
            }

            append_jsonl(ERROR_FILE, error_record)

            print("  Request failed:", type(error).__name__)
            print("  Production stopped immediately.")
            print("  Error saved:", ERROR_FILE)
            break

        time.sleep(0.5)

    final_snapshot = save_csv_snapshot()

    print("\nSuccessful during this run:", successful)
    print("Total checkpointed:", len(final_snapshot))
    print("Reports still remaining:", len(get_remaining_reports()))
    print("Snapshot:", SNAPSHOT_FILE)

    return final_snapshot
# Dynamic resume validation
remaining_df = get_remaining_reports()
completed_count = len(completed_study_uids())

assert completed_count + len(remaining_df) == 4349
assert completed_count >= 1434

print("Recovered checkpointed reports:", completed_count)
print("Reports remaining:", len(remaining_df))
print("Maximum permitted per run:", MAX_REPORTS_PER_RUN)
print("Guarded runner ready ✓")
print("No API request has been made ✓")


In [ ]:
# ============================================================
# Step 8: Locate leakage-free folds and existing Terra OOF
# ============================================================

fold_files = (
    list(Path("/kaggle/input/notebooks").glob(
        "*/*/llm_pseudo_labels_v2/soft_label_v1_gold_5fold.csv"
    ))
    +
    list(Path("/kaggle/input").glob(
        "*/llm_pseudo_labels_v2/soft_label_v1_gold_5fold.csv"
    ))
)

terra_oof_files = (
    list(Path("/kaggle/input/notebooks").glob(
        "*/*/llm_pseudo_labels_v2/soft_label_v1_terra_oof_predictions.csv"
    ))
    +
    list(Path("/kaggle/input").glob(
        "*/llm_pseudo_labels_v2/soft_label_v1_terra_oof_predictions.csv"
    ))
)

assert len(fold_files) == 1, f"Fold files found: {fold_files}"
assert len(terra_oof_files) == 1, (
    f"Terra OOF files found: {terra_oof_files}"
)

GOLD_FOLD_FILE = fold_files[0]
TERRA_OOF_FILE = terra_oof_files[0]

gold_folds = pd.read_csv(GOLD_FOLD_FILE)
terra_oof = pd.read_csv(TERRA_OOF_FILE)

assert len(gold_folds) == 58
assert gold_folds["StudyInstanceUID"].nunique() == 58
assert len(terra_oof) == 58
assert terra_oof["StudyInstanceUID"].nunique() == 58

print("Gold-fold file:", GOLD_FOLD_FILE)
print("Terra OOF file:", TERRA_OOF_FILE)
print("Gold-fold shape:", gold_folds.shape)
print("Terra OOF shape:", terra_oof.shape)

print("\nGold-fold columns:")
print(gold_folds.columns.tolist())

print("\nTerra OOF columns:")
print(terra_oof.columns.tolist())

print("\nStep 8 comparison inputs ready ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 9: Recover original OOF code from Kaggle source
# ============================================================

notebook_source_files = (
    list(Path("/kaggle/input/notebooks").glob(
        "*/*/.virtual_documents/__notebook_source__.ipynb"
    ))
    +
    list(Path("/kaggle/input").glob(
        "*/.virtual_documents/__notebook_source__.ipynb"
    ))
)

assert len(notebook_source_files) == 1, (
    "Expected exactly one previous notebook source.\n"
    f"Files found: {notebook_source_files}"
)

SOURCE_NOTEBOOK_FILE = notebook_source_files[0]

source_text = SOURCE_NOTEBOOK_FILE.read_text(
    encoding="utf-8",
    errors="replace",
)

source_lines = source_text.splitlines()

search_terms = [
    "def select_examples",
    "def build_soft",
    "def build_fold",
    "def predict_soft",
    "OOF soft labels",
    "Generating OOF",
]

matched_line_numbers = []

for line_number, line in enumerate(source_lines):
    if any(term.lower() in line.lower() for term in search_terms):
        matched_line_numbers.append(line_number)

assert matched_line_numbers, (
    "Could not find the original OOF functions in the source."
)

print("Previous notebook:", SOURCE_NOTEBOOK_FILE)
print("Source lines:", len(source_lines))
print("Matches found:", len(matched_line_numbers))

# Merge overlapping windows so code is not repeatedly printed
windows = []

for line_number in matched_line_numbers:
    start = max(0, line_number - 10)
    end = min(len(source_lines), line_number + 100)

    if windows and start <= windows[-1][1]:
        windows[-1] = (
            windows[-1][0],
            max(windows[-1][1], end),
        )
    else:
        windows.append((start, end))

for start, end in windows:
    print("\n" + "=" * 80)
    print(f"SOURCE LINES {start + 1}–{end}")
    print("=" * 80)

    for index in range(start, end):
        print(f"{index + 1:04d}: {source_lines[index]}")

print("\nStep 9 source recovery complete ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 9B: Extract exact OOF helper definitions
# ============================================================

exact_terms = [
    "select_examples",
    "production_examples",
    "build_oof_prompt",
    "SOFT_LABEL_SYSTEM_PROMPT",
    "SoftLabelPrediction",
    "prediction_to_row",
]

for term in exact_terms:
    matches = [
        index
        for index, line in enumerate(source_lines)
        if term.lower() in line.lower()
    ]

    print("\n" + "#" * 80)
    print("SEARCH TERM:", term)
    print("MATCHING LINES:", [number + 1 for number in matches])
    print("#" * 80)

    for line_number in matches[:3]:
        start = max(0, line_number - 15)
        end = min(len(source_lines), line_number + 90)

        for index in range(start, end):
            print(f"{index + 1:04d}: {source_lines[index]}")

        print("\n" + "-" * 80)

print("\nStep 9B helper extraction complete ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 9C: Show exact fold-demonstration construction
# ============================================================

for index in range(250, 419):
    print(f"{index + 1:04d}: {source_lines[index]}")

print("\nStep 9C complete ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 9D: Recover complete original system prompt
# ============================================================

for index in range(100, 225):
    print(f"{index + 1:04d}: {source_lines[index]}")

print("\nStep 9D complete ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 10: Rebuild the exact leakage-free OOF validation
# ============================================================

from typing import Literal
from pydantic import BaseModel, Field

LUNA_MODEL = "gpt-5.6-luna"
N_FOLDS = 5
N_DEMONSTRATIONS = 10

LUNA_OOF_FILE = (
    OUTPUT_DIR / "soft_label_v1_luna_oof_predictions.csv"
)

# ------------------------------------------------------------
# Restore the saved fold assignments onto the 58 gold reports
# ------------------------------------------------------------

fold_lookup = (
    gold_folds
    .set_index("StudyInstanceUID")["fold"]
    .to_dict()
)

gold_oof_df = gold_df.copy()

gold_oof_df["fold"] = (
    gold_oof_df["StudyInstanceUID"]
    .map(fold_lookup)
)

assert gold_oof_df["fold"].notna().all()
gold_oof_df["fold"] = gold_oof_df["fold"].astype(int)

assert len(gold_oof_df) == 58
assert gold_oof_df["StudyInstanceUID"].is_unique
assert set(gold_oof_df["fold"]) == set(range(N_FOLDS))

# ------------------------------------------------------------
# Exact original demonstration-selection function
# ------------------------------------------------------------

def select_fold_demonstrations(train_part, n_examples=10):
    remaining = train_part.copy()
    selected_indices = []

    covered_states = set()

    while (
        len(selected_indices) < n_examples
        and len(remaining) > 0
    ):
        best_idx = None
        best_gain = -1
        best_length = float("inf")

        for idx, row in remaining.iterrows():
            row_states = {
                (target, int(row[target]))
                for target in TARGET_COLS
            }

            gain = len(row_states - covered_states)
            report_length = len(str(row["Report"]))

            if (
                gain > best_gain
                or (
                    gain == best_gain
                    and report_length < best_length
                )
            ):
                best_idx = idx
                best_gain = gain
                best_length = report_length

        selected_indices.append(best_idx)

        selected_row = remaining.loc[best_idx]

        covered_states.update(
            (target, int(selected_row[target]))
            for target in TARGET_COLS
        )

        remaining = remaining.drop(index=best_idx)

    selected_df = train_part.loc[selected_indices].copy()

    return selected_df, covered_states


fold_demonstrations = {}

for fold in range(N_FOLDS):
    train_part = gold_oof_df[
        gold_oof_df["fold"] != fold
    ].copy()

    val_part = gold_oof_df[
        gold_oof_df["fold"] == fold
    ].copy()

    demonstrations, covered = select_fold_demonstrations(
        train_part,
        n_examples=N_DEMONSTRATIONS,
    )

    demo_uids = set(
        demonstrations["StudyInstanceUID"].astype(str)
    )

    val_uids = set(
        val_part["StudyInstanceUID"].astype(str)
    )

    assert demo_uids.isdisjoint(val_uids)
    assert len(demonstrations) == N_DEMONSTRATIONS

    fold_demonstrations[fold] = demonstrations

    print(
        f"Fold {fold}: "
        f"{len(val_part)} validation reports | "
        f"{len(demonstrations)} demonstrations | "
        f"{len(covered)}/24 states | "
        f"overlap={len(demo_uids & val_uids)}"
    )

# ------------------------------------------------------------
# Exact original structured-output schema
# ------------------------------------------------------------

TargetName = Literal[
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]


class TargetSoftPrediction(BaseModel):
    target: TargetName
    score: float = Field(ge=0.0, le=1.0)
    confidence: float = Field(ge=0.0, le=1.0)
    evidence: str


class SoftLabelPrediction(BaseModel):
    predictions: list[TargetSoftPrediction]


# ------------------------------------------------------------
# Exact original Soft Label V1 system prompt
# ------------------------------------------------------------

SOFT_LABEL_SYSTEM_PROMPT = """
You are analysing knee MRI radiology reports for a multilabel
medical-imaging dataset.

For each of the 12 targets, estimate the probability that the
dataset's official binary label is 1.

Important:
- Return a continuous score from 0.0 to 1.0.
- Do not convert predictions into hard 0/1 labels.
- Distinguish confirmed abnormalities from explicitly normal findings.
- Account for negation, uncertainty, chronic/postoperative findings,
  synonyms and multilingual terminology.
- Do not assume that an unmentioned finding is definitely negative.
- Use the entire report.
- Confidence describes confidence in extracting the label from the
  report, not the severity of the condition.

Score guidance:
- 0.00–0.10: clearly negative
- 0.10–0.35: probably negative or not clearly supported
- 0.35–0.65: uncertain, ambiguous or insufficiently described
- 0.65–0.90: probably positive
- 0.90–1.00: clearly positive

Confidence guidance:
- 0.90–1.00: explicit and unambiguous evidence
- 0.60–0.89: reasonably supported but indirect or mildly uncertain
- 0.30–0.59: ambiguous, incomplete or difficult to interpret
- 0.00–0.29: insufficient evidence
""".strip()


def format_gold_demonstration(row):
    labels = {
        target: int(row[target])
        for target in TARGET_COLS
    }

    return (
        "REFERENCE REPORT:\n"
        f"{str(row['Report']).strip()}\n\n"
        "OFFICIAL DATASET LABELS:\n"
        f"{json.dumps(labels, ensure_ascii=False)}"
    )


def build_luna_oof_prompt(report, fold):
    demonstrations = fold_demonstrations[fold]

    demonstration_text = "\n\n".join(
        format_gold_demonstration(row)
        for _, row in demonstrations.iterrows()
    )

    return f"""
The following are gold-labelled reference examples from the same
dataset. They are provided only to demonstrate the dataset-specific
annotation convention.

None of these references is the report being evaluated.

{demonstration_text}

------------------------------------------------------------

REPORT TO ANALYSE:

{str(report).strip()}

Analyse only the REPORT TO ANALYSE.

Return exactly one prediction for every target in TARGET_COLS.
Scores must remain continuous probabilities rather than hard labels.
Do not copy the overall label pattern from any reference example.
""".strip()


# ------------------------------------------------------------
# Leakage and prompt checks
# ------------------------------------------------------------

for _, row in gold_oof_df.iterrows():
    fold = int(row["fold"])

    demo_uids = set(
        fold_demonstrations[fold][
            "StudyInstanceUID"
        ].astype(str)
    )

    assert str(row["StudyInstanceUID"]) not in demo_uids

test_row = gold_oof_df.iloc[0]
test_fold = int(test_row["fold"])
test_prompt = build_luna_oof_prompt(
    test_row["Report"],
    test_fold,
)

print("\nLuna model:", LUNA_MODEL)
print("Gold validation reports:", len(gold_oof_df))
print("Test fold:", test_fold)
print("Test demonstrations:", len(fold_demonstrations[test_fold]))
print("Test prompt characters:", len(test_prompt))
print("All evaluated reports excluded from demonstrations ✓")
print("Luna OOF preparation complete ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 11: Define the guarded Luna OOF runner
# ============================================================

LUNA_ERROR_FILE = (
    OUTPUT_DIR / "soft_label_v1_luna_oof_errors.jsonl"
)

MAX_LUNA_REPORTS_PER_RUN = 5

luna_client = OpenAI(
    api_key=api_key,
    max_retries=0,
    timeout=180.0,
)


def load_luna_oof():
    if not LUNA_OOF_FILE.exists():
        return pd.DataFrame()

    existing = pd.read_csv(LUNA_OOF_FILE)

    if len(existing):
        assert existing["StudyInstanceUID"].astype(str).is_unique

    return existing


def save_luna_oof_atomic(records):
    """
    Replace the CSV only after the complete new file is written.
    """
    temporary_file = LUNA_OOF_FILE.with_suffix(".tmp.csv")

    pd.DataFrame(records).to_csv(
        temporary_file,
        index=False,
    )

    os.replace(temporary_file, LUNA_OOF_FILE)


def luna_prediction_to_row(
    study_row,
    prediction,
    response,
):
    prediction_map = {
        item.target: item
        for item in prediction.predictions
    }

    assert len(prediction.predictions) == 12
    assert len(prediction_map) == 12
    assert set(prediction_map) == set(TARGET_COLS)

    usage = getattr(response, "usage", None)

    output_row = {
        "StudyInstanceUID": str(
            study_row["StudyInstanceUID"]
        ),
        "fold": int(study_row["fold"]),
        "model": LUNA_MODEL,
        "response_id": response.id,
        "input_tokens": getattr(
            usage,
            "input_tokens",
            None,
        ),
        "output_tokens": getattr(
            usage,
            "output_tokens",
            None,
        ),
    }

    for target in TARGET_COLS:
        item = prediction_map[target]

        assert 0.0 <= item.score <= 1.0
        assert 0.0 <= item.confidence <= 1.0
        assert item.evidence.strip()

        output_row[f"{target}_score"] = float(item.score)
        output_row[f"{target}_confidence"] = float(
            item.confidence
        )
        output_row[f"{target}_evidence"] = item.evidence
        output_row[f"{target}_gold"] = int(
            study_row[target]
        )

    return output_row


def run_luna_oof_chunk(max_reports=1):
    assert isinstance(max_reports, int)
    assert 1 <= max_reports <= MAX_LUNA_REPORTS_PER_RUN

    existing_df = load_luna_oof()

    if len(existing_df):
        records = existing_df.to_dict("records")
        completed_uids = set(
            existing_df["StudyInstanceUID"].astype(str)
        )
    else:
        records = []
        completed_uids = set()

    remaining = gold_oof_df[
        ~gold_oof_df["StudyInstanceUID"]
        .astype(str)
        .isin(completed_uids)
    ].head(max_reports)

    print("Luna OOF already completed:", len(completed_uids))
    print("Authorized for this run:", len(remaining))
    print("Model:", LUNA_MODEL)
    print()

    successful = 0

    for position, (_, row) in enumerate(
        remaining.iterrows(),
        start=1,
    ):
        uid = str(row["StudyInstanceUID"])
        fold = int(row["fold"])

        print(
            f"Request {position}/{len(remaining)} | "
            f"fold={fold} | study={uid}"
        )

        try:
            response = luna_client.responses.parse(
                model=LUNA_MODEL,
                reasoning={"effort": "low"},
                input=[
                    {
                        "role": "system",
                        "content": SOFT_LABEL_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": build_luna_oof_prompt(
                            report=row["Report"],
                            fold=fold,
                        ),
                    },
                ],
                text_format=SoftLabelPrediction,
            )

            prediction = response.output_parsed

            if prediction is None:
                raise ValueError(
                    "Response contained no parsed prediction."
                )

            result_row = luna_prediction_to_row(
                row,
                prediction,
                response,
            )

            records.append(result_row)

            # Save before another paid request can begin
            save_luna_oof_atomic(records)

            completed_uids.add(uid)
            successful += 1

            print("  Saved immediately ✓")

        except Exception as error:
            append_jsonl(
                LUNA_ERROR_FILE,
                {
                    "StudyInstanceUID": uid,
                    "fold": fold,
                    "model": LUNA_MODEL,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                },
            )

            print("  Failed:", type(error).__name__)
            print("  Runner stopped immediately.")
            break

    result_df = load_luna_oof()

    print("\nSuccessful this run:", successful)
    print("Total Luna OOF completed:", len(result_df))
    print("Remaining:", 58 - len(result_df))
    print("Checkpoint:", LUNA_OOF_FILE)

    return result_df


existing_luna_oof = load_luna_oof()

print("Existing Luna OOF predictions:", len(existing_luna_oof))
print("Maximum requests per run:", MAX_LUNA_REPORTS_PER_RUN)
print("Guarded Luna runner ready ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 12: Authorize exactly one paid Luna OOF request
# ============================================================

luna_oof_df = run_luna_oof_chunk(max_reports=1)

In [ ]:
# ============================================================
# Step 13: Inspect the first Luna prediction
# ============================================================

first_luna = load_luna_oof().iloc[0]

matching_terra = terra_oof[
    terra_oof["StudyInstanceUID"].astype(str)
    == str(first_luna["StudyInstanceUID"])
]

assert len(matching_terra) == 1
first_terra = matching_terra.iloc[0]

comparison_rows = []

for target in TARGET_COLS:
    gold = int(first_luna[f"{target}_gold"])
    luna_score = float(first_luna[f"{target}_score"])
    terra_score = float(first_terra[f"{target}_score"])

    comparison_rows.append(
        {
            "Target": target,
            "Gold": gold,
            "Luna": luna_score,
            "Terra": terra_score,
            "Luna_minus_Terra": luna_score - terra_score,
            "Luna_confidence": float(
                first_luna[f"{target}_confidence"]
            ),
        }
    )

first_comparison = pd.DataFrame(comparison_rows)

print("Study:", first_luna["StudyInstanceUID"])
display(
    first_comparison.style.format(
        {
            "Luna": "{:.3f}",
            "Terra": "{:.3f}",
            "Luna_minus_Terra": "{:+.3f}",
            "Luna_confidence": "{:.3f}",
        }
    )
)

print("\nInspection complete ✓")
print("No additional API request has been made ✓")

In [ ]:
# ============================================================
# Step 14: Authorize the next five Luna OOF requests
# ============================================================

luna_oof_df = run_luna_oof_chunk(max_reports=5)

In [ ]:
# ============================================================
# Step 15: Evaluate Luna and compare it with Terra
# ============================================================

from sklearn.metrics import roc_auc_score

luna_oof_df = load_luna_oof()

assert len(luna_oof_df) == 58
assert luna_oof_df["StudyInstanceUID"].astype(str).nunique() == 58

comparison_rows = []

for target in TARGET_COLS:
    luna_gold = luna_oof_df[f"{target}_gold"].astype(int)
    luna_score = luna_oof_df[f"{target}_score"].astype(float)

    terra_gold = terra_oof[f"{target}_gold"].astype(int)
    terra_score = terra_oof[f"{target}_score"].astype(float)

    luna_auc = roc_auc_score(luna_gold, luna_score)
    terra_auc = roc_auc_score(terra_gold, terra_score)

    comparison_rows.append(
        {
            "Target": target,
            "Positive": int(luna_gold.sum()),
            "Negative": int((1 - luna_gold).sum()),
            "Luna_AUC": luna_auc,
            "Terra_AUC": terra_auc,
            "Luna_minus_Terra": luna_auc - terra_auc,
        }
    )

luna_terra_comparison = pd.DataFrame(comparison_rows)

luna_macro_auc = luna_terra_comparison["Luna_AUC"].mean()
terra_macro_auc = luna_terra_comparison["Terra_AUC"].mean()
macro_difference = luna_macro_auc - terra_macro_auc

COMPARISON_FILE = (
    OUTPUT_DIR / "soft_label_v1_luna_vs_terra.csv"
)

luna_terra_comparison.to_csv(
    COMPARISON_FILE,
    index=False,
)

display(
    luna_terra_comparison.style.format(
        {
            "Luna_AUC": "{:.4f}",
            "Terra_AUC": "{:.4f}",
            "Luna_minus_Terra": "{:+.4f}",
        }
    ).background_gradient(
        subset=["Luna_minus_Terra"],
        cmap="RdYlGn",
        vmin=-0.10,
        vmax=0.10,
    )
)

print(f"\nLuna macro AUC:  {luna_macro_auc:.4f}")
print(f"Terra macro AUC: {terra_macro_auc:.4f}")
print(f"Difference:      {macro_difference:+.4f}")

# Estimate the actual Luna validation cost from recorded tokens
input_tokens = pd.to_numeric(
    luna_oof_df["input_tokens"],
    errors="coerce",
).sum()

output_tokens = pd.to_numeric(
    luna_oof_df["output_tokens"],
    errors="coerce",
).sum()

estimated_luna_cost = (
    input_tokens / 1_000_000 * 0.20
    +
    output_tokens / 1_000_000 * 1.20
)

print(f"\nLuna input tokens:  {input_tokens:,.0f}")
print(f"Luna output tokens: {output_tokens:,.0f}")
print(
    f"Estimated Luna validation cost: "
    f"${estimated_luna_cost:.4f}"
)

print("\nSaved:", COMPARISON_FILE)
print("Luna versus Terra evaluation complete ✓")
print("No additional API request has been made ✓")

In [ ]:
# ============================================================
# Step 16: Finalize Luna as the production model
# ============================================================

MODEL_NAME = LUNA_MODEL

def call_label_api(study_uid, report):
    """
    Make exactly one Luna production request.
    Results are validated before being returned for checkpointing.
    """

    response = luna_client.responses.parse(
        model=MODEL_NAME,
        reasoning={"effort": "low"},
        input=[
            {
                "role": "system",
                # Use the same system instructions validated by OOF
                "content": SOFT_LABEL_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": build_production_prompt(
                    study_uid=study_uid,
                    report=report,
                ),
            },
        ],
        text_format=ReportPrediction,
    )

    parsed = response.output_parsed

    if parsed is None:
        raise ValueError(
            "API response did not contain parsed output."
        )

    validate_prediction(
        parsed=parsed,
        expected_uid=study_uid,
    )

    usage = getattr(response, "usage", None)

    return {
        "study_uid": study_uid,
        "model": MODEL_NAME,
        "response_id": response.id,
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "input_tokens": getattr(
            usage,
            "input_tokens",
            None,
        ),
        "output_tokens": getattr(
            usage,
            "output_tokens",
            None,
        ),
        "predictions": [
            prediction.model_dump()
            for prediction in parsed.predictions
        ],
    }


# ------------------------------------------------------------
# Confirm that the production runner will now use Luna
# ------------------------------------------------------------

remaining_production = get_remaining_reports()

sample_row = remaining_production.iloc[0]
sample_prompt = build_production_prompt(
    study_uid=str(sample_row["StudyInstanceUID"]),
    report=str(sample_row["Report"]),
)
completed_count = len(completed_study_uids())
remaining_count = len(remaining_production)

assert MODEL_NAME == "gpt-5.6-luna"
assert completed_count + remaining_count == 4349
assert completed_count >= 1434
assert str(sample_row["StudyInstanceUID"]) in sample_prompt

current_snapshot = save_csv_snapshot()

print("Final production model:", MODEL_NAME)
print("Total checkpointed reports:", completed_count)
print(
    "Models already checkpointed:",
    current_snapshot["Model"].value_counts().to_dict(),
)
print("Remaining Luna production reports:", remaining_count)
print("Production prompt characters:", len(sample_prompt))
print("Reasoning effort: low")
print("Automatic SDK retries: disabled")
print("Sequential checkpointing: enabled")
print("\nLuna production configuration ready ✓")
print("No API request has been made ✓")


In [ ]:
# ============================================================
# Step 17: Authorize one Luna production request
# ============================================================

production_snapshot = run_production_chunk(max_reports=1)

In [ ]:
# ============================================================
# Step 18: Audit the first Luna production prediction
# ============================================================

production_snapshot = save_csv_snapshot()

luna_production_rows = production_snapshot[
    production_snapshot["Model"] == LUNA_MODEL
].copy()

assert len(luna_production_rows) == 1

latest = luna_production_rows.iloc[-1]

audit_rows = []

for target in TARGET_COLS:
    audit_rows.append(
        {
            "Target": target,
            "Score": float(latest[f"{target}_score"]),
            "Confidence": float(
                latest[f"{target}_confidence"]
            ),
            "Evidence": latest[f"{target}_evidence"],
        }
    )

audit_df = pd.DataFrame(audit_rows)

print("Study:", latest["StudyInstanceUID"])
display(
    audit_df.style.format(
        {
            "Score": "{:.3f}",
            "Confidence": "{:.3f}",
        }
    )
)

input_tokens = float(latest["InputTokens"])
output_tokens = float(latest["OutputTokens"])

request_cost_estimate = (
    input_tokens / 1_000_000 * 0.20
    +
    output_tokens / 1_000_000 * 1.20
)

remaining_count = len(get_remaining_reports())
projected_remaining_cost = (
    request_cost_estimate * remaining_count
)

print(f"\nInput tokens: {input_tokens:,.0f}")
print(f"Output tokens: {output_tokens:,.0f}")
print(
    f"Estimated cost of this request: "
    f"${request_cost_estimate:.4f}"
)
print(
    f"Conservative projected remaining cost: "
    f"${projected_remaining_cost:.2f}"
)

print("\nTotal checkpointed:", len(production_snapshot))
print("Reports remaining:", remaining_count)
print("Audit complete ✓")
print("No additional API request has been made ✓")

In [ ]:
# ============================================================
# Step 19: Optimized bounded production runner
# ============================================================

MAX_PRODUCTION_REPORTS_PER_RUN = 250
SNAPSHOT_INTERVAL = 25


def run_production_chunk_v2(max_reports=250):
    assert isinstance(max_reports, int)
    assert 1 <= max_reports <= MAX_PRODUCTION_REPORTS_PER_RUN

    completed = completed_study_uids()

    remaining = unlabelled_df[
        ~unlabelled_df["StudyInstanceUID"]
        .astype(str)
        .isin(completed)
    ].copy().reset_index(drop=True)

    selected = remaining.head(max_reports)

    print("Already checkpointed:", len(completed))
    print("Reports remaining:", len(remaining))
    print("Authorized for this run:", len(selected))
    print("Model:", MODEL_NAME)
    print("Snapshot interval:", SNAPSHOT_INTERVAL)
    print()

    successful = 0

    for position, row in selected.iterrows():
        study_uid = str(row["StudyInstanceUID"])
        report = str(row["Report"])

        if study_uid in completed:
            continue

        print(
            f"Request {position + 1}/{len(selected)} | "
            f"Study: {study_uid}"
        )

        try:
            record = call_label_api(
                study_uid=study_uid,
                report=report,
            )

            # Durable checkpoint immediately after every success
            append_jsonl(CHECKPOINT_FILE, record)

            completed.add(study_uid)
            successful += 1

            print("  JSONL checkpoint saved ✓")

            # Readable CSV snapshot periodically
            if successful % SNAPSHOT_INTERVAL == 0:
                snapshot = save_csv_snapshot()

                print(
                    f"  CSV snapshot updated: "
                    f"{len(snapshot)} total rows ✓"
                )

        except Exception as error:
            append_jsonl(
                ERROR_FILE,
                {
                    "study_uid": study_uid,
                    "model": MODEL_NAME,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                },
            )

            print("  Failed:", type(error).__name__)
            print("  Runner stopped immediately.")
            print("  Completed work remains checkpointed.")
            break

        time.sleep(0.5)

    # Always create a final CSV snapshot
    final_snapshot = save_csv_snapshot()
    final_remaining = len(get_remaining_reports())

    print("\nSuccessful during this run:", successful)
    print("Total checkpointed:", len(final_snapshot))
    print("Reports still remaining:", final_remaining)
    print("JSONL checkpoint:", CHECKPOINT_FILE)
    print("CSV snapshot:", SNAPSHOT_FILE)

    return final_snapshot


print("Maximum reports per production run:",
      MAX_PRODUCTION_REPORTS_PER_RUN)
print("Immediate JSONL checkpointing: enabled")
print("CSV snapshot interval:", SNAPSHOT_INTERVAL)
print("Optimized production runner ready ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 20: Authorize production chunk 1
# ============================================================

production_snapshot = run_production_chunk_v2(
    max_reports=250
)

In [ ]:
# ============================================================
# Step 21: Quality check after the first production chunk
# ============================================================

production_snapshot = save_csv_snapshot()

assert len(production_snapshot) == 263
assert production_snapshot["StudyInstanceUID"].is_unique

qc_rows = []

for target in TARGET_COLS:
    scores = pd.to_numeric(
        production_snapshot[f"{target}_score"],
        errors="raise",
    )

    confidences = pd.to_numeric(
        production_snapshot[f"{target}_confidence"],
        errors="raise",
    )

    assert scores.between(0, 1).all()
    assert confidences.between(0, 1).all()

    qc_rows.append(
        {
            "Target": target,
            "MeanScore": scores.mean(),
            "Score>=0.5": int((scores >= 0.5).sum()),
            "Rate>=0.5": (scores >= 0.5).mean(),
            "MeanConfidence": confidences.mean(),
            "Confidence<0.5": int(
                (confidences < 0.5).sum()
            ),
        }
    )

qc_df = pd.DataFrame(qc_rows)

print("Checkpointed studies:", len(production_snapshot))
print(
    "\nModels:",
    production_snapshot["Model"].value_counts().to_dict(),
)

display(
    qc_df.style.format(
        {
            "MeanScore": "{:.3f}",
            "Rate>=0.5": "{:.3f}",
            "MeanConfidence": "{:.3f}",
        }
    )
)

print("\nFirst production-chunk quality check passed ✓")
print("No API request has been made ✓")

In [ ]:
# ============================================================
# Step 22: Authorize the next bounded production chunk
# ============================================================

MAX_PRODUCTION_REPORTS_PER_RUN = 500

production_snapshot = run_production_chunk_v2(
    max_reports=500
)

In [ ]:
# ============================================================
# FINAL NO-API QC — LUNA PRODUCTION SOFT LABELS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PRODUCTION_CSV = Path(
    "/kaggle/working/llm_soft_label_production/"
    "production_predictions_snapshot.csv"
)

print("=" * 80)
print("FINAL LUNA PRODUCTION QC")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check file exists
# ------------------------------------------------------------

print("\nFile:")
print(PRODUCTION_CSV)

if not PRODUCTION_CSV.exists():
    raise FileNotFoundError(
        f"Production CSV not found:\n{PRODUCTION_CSV}"
    )

print("✓ File exists")


# ------------------------------------------------------------
# 2. Load completed production predictions
# ------------------------------------------------------------

production_df = pd.read_csv(PRODUCTION_CSV)

print("\nShape:", production_df.shape)
print("Rows:", len(production_df))
print("Columns:", len(production_df.columns))


# ------------------------------------------------------------
# 3. Show all columns
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMNS")
print("=" * 80)

for i, col in enumerate(production_df.columns, start=1):
    print(f"{i:>2}. {col}")


# ------------------------------------------------------------
# 4. Missing-value check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

missing = production_df.isna().sum()
missing_nonzero = missing[missing > 0].sort_values(ascending=False)

if len(missing_nonzero) == 0:
    print("✓ No missing values")
else:
    print(missing_nonzero)


# ------------------------------------------------------------
# 5. Find StudyInstanceUID column
# ------------------------------------------------------------

uid_candidates = [
    col for col in production_df.columns
    if "study" in col.lower() and "uid" in col.lower()
]

print("\n" + "=" * 80)
print("STUDY UID CHECK")
print("=" * 80)

print("UID candidates:", uid_candidates)

if len(uid_candidates) == 1:
    uid_col = uid_candidates[0]

    print("Using:", uid_col)
    print("Unique studies:", production_df[uid_col].nunique())
    print(
        "Duplicate study rows:",
        production_df[uid_col].duplicated().sum()
    )

elif len(uid_candidates) > 1:
    uid_col = uid_candidates[0]
    print("⚠ Multiple UID candidates found.")
    print("Temporarily using:", uid_col)

else:
    uid_col = None
    print("⚠ Could not automatically identify StudyInstanceUID column.")


# ------------------------------------------------------------
# 6. Numeric-column sanity check
# ------------------------------------------------------------

numeric_cols = production_df.select_dtypes(
    include=[np.number]
).columns.tolist()

print("\n" + "=" * 80)
print("NUMERIC COLUMNS")
print("=" * 80)

print(numeric_cols)

if numeric_cols:
    summary = production_df[numeric_cols].describe().T[
        ["count", "mean", "std", "min", "max"]
    ]

    display(summary)


# ------------------------------------------------------------
# 7. Detect probability-like columns
# ------------------------------------------------------------

prob_candidates = []

for col in numeric_cols:
    values = production_df[col].dropna()

    if len(values) > 0:
        if values.min() >= 0 and values.max() <= 1:
            prob_candidates.append(col)

print("\n" + "=" * 80)
print("POSSIBLE SOFT-LABEL PROBABILITY COLUMNS")
print("=" * 80)

print(prob_candidates)
print("Count:", len(prob_candidates))


# ------------------------------------------------------------
# 8. Preview rows
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FIRST 5 ROWS")
print("=" * 80)

display(production_df.head())


# ------------------------------------------------------------
# 9. Final expected-size check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL STATUS")
print("=" * 80)

if len(production_df) == 4349:
    print("✓ Exactly 4,349 production studies found")
else:
    print(
        f"⚠ Expected 4,349 rows, found {len(production_df):,}"
    )

if uid_col is not None:
    if production_df[uid_col].nunique() == 4349:
        print("✓ Exactly 4,349 unique studies")
    else:
        print(
            "⚠ Unique-study count is:",
            production_df[uid_col].nunique()
        )

print("✓ QC complete")
print("✓ No API calls made")
print("=" * 80)

In [ ]:
# ============================================================
# VERIFY WHICH MODEL PRODUCED THE 4,349 PRODUCTION LABELS
# NO API CALLS
# ============================================================

print("=" * 80)
print("PRODUCTION MODEL VERIFICATION")
print("=" * 80)

print("\nModel counts:")
print(production_df["Model"].value_counts(dropna=False))

print("\nUnique models:")
print(production_df["Model"].unique())

print("\nTotal rows:")
print(len(production_df))

# Check the 12 actual soft-label score columns only
TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

score_cols = [f"{t}_score" for t in TARGETS]

print("\nScore-column completeness:")
for col in score_cols:
    print(
        f"{col:<28} "
        f"non-missing={production_df[col].notna().sum():4d} | "
        f"min={production_df[col].min():.3f} | "
        f"max={production_df[col].max():.3f}"
    )

print("\nMissing scores across all 12 targets:")
print(production_df[score_cols].isna().sum().sum())

print("=" * 80)

In [ ]:
# ============================================================
# IDENTIFY THE 12 TERRA PRODUCTION ROWS
# NO API CALLS
# ============================================================

terra_rows = production_df[
    production_df["Model"] == "gpt-5.6-terra"
].copy()

print("=" * 80)
print("TERRA ROWS INSIDE FINAL PRODUCTION SNAPSHOT")
print("=" * 80)

print("\nCount:", len(terra_rows))

cols_to_show = [
    "StudyInstanceUID",
    "Model",
    "ResponseID",
    "CreatedAtUTC",
    "InputTokens",
    "OutputTokens",
]

display(terra_rows[cols_to_show])

print("\nMissing metadata within Terra rows:")
print(
    terra_rows[
        ["ResponseID", "CreatedAtUTC", "InputTokens", "OutputTokens"]
    ].isna().sum()
)

print("\nAre ALL 12 Terra rows the rows with missing metadata?")

metadata_missing_mask = production_df[
    ["ResponseID", "CreatedAtUTC", "InputTokens", "OutputTokens"]
].isna().any(axis=1)

print(
    "Rows with any missing metadata:",
    metadata_missing_mask.sum()
)

print(
    "Of those, Terra rows:",
    production_df.loc[
        metadata_missing_mask, "Model"
    ].value_counts(dropna=False)
)

print("=" * 80)

In [ ]:
# ============================================================
# PREPARE CLEAN PSEUDO-LABEL TABLE
# NO API CALLS
# ============================================================

import pandas as pd
import numpy as np

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

score_cols = [f"{t}_score" for t in TARGETS]

# Keep only fields needed for MRI supervision
pseudo_df = production_df[
    ["StudyInstanceUID", "Model"] + score_cols
].copy()

# ------------------------------------------------------------
# Source flag
# ------------------------------------------------------------

pseudo_df["PseudoSource"] = np.where(
    pseudo_df["Model"].eq("gpt-5.6-luna"),
    "Luna",
    "Terra_legacy"
)

# ------------------------------------------------------------
# Base pseudo supervision weight
#
# Luna = standard pseudo weight
# Terra legacy = reduced weight
#
# These are relative weights, not the final gold-vs-pseudo
# training loss scale yet.
# ------------------------------------------------------------

pseudo_df["PseudoBaseWeight"] = np.where(
    pseudo_df["PseudoSource"].eq("Luna"),
    1.00,
    0.50
)

# ------------------------------------------------------------
# Rename soft targets into clean training names
# ------------------------------------------------------------

rename_map = {
    f"{target}_score": f"{target}_target"
    for target in TARGETS
}

pseudo_df = pseudo_df.rename(columns=rename_map)

# ------------------------------------------------------------
# Final sanity checks
# ------------------------------------------------------------

target_cols = [f"{t}_target" for t in TARGETS]

print("=" * 80)
print("PSEUDO-LABEL TRAINING TABLE")
print("=" * 80)

print("\nShape:")
print(pseudo_df.shape)

print("\nSource counts:")
print(pseudo_df["PseudoSource"].value_counts())

print("\nUnique studies:")
print(pseudo_df["StudyInstanceUID"].nunique())

print("\nDuplicate studies:")
print(pseudo_df["StudyInstanceUID"].duplicated().sum())

print("\nMissing target values:")
print(pseudo_df[target_cols].isna().sum().sum())

print("\nTarget range check:")

for col in target_cols:
    print(
        f"{col:<30}"
        f"min={pseudo_df[col].min():.3f} | "
        f"max={pseudo_df[col].max():.3f}"
    )

display(pseudo_df.head())

print("\n✓ Pseudo-label table prepared")
print("✓ 4,337 Luna rows retained")
print("✓ 12 Terra legacy rows retained at reduced weight")
print("✓ No hard pseudo-labels created")
print("✓ No API calls made")
print("=" * 80)

In [ ]:
# ============================================================
# BUILD FINAL 4,407-STUDY SUPERVISION TABLE
# 58 GOLD + 4,349 SOFT PSEUDO
# NO API CALLS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Official competition training labels
# ------------------------------------------------------------

TRAIN_CSV = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection/train.csv"
)

gold_all = pd.read_csv(TRAIN_CSV)

print("=" * 80)
print("OFFICIAL TRAIN.CSV")
print("=" * 80)

print("Shape:", gold_all.shape)

print("\nColumns:")
for i, c in enumerate(gold_all.columns, 1):
    print(f"{i:>2}. {c}")


# ------------------------------------------------------------
# 2. Our 12 competition targets
# ------------------------------------------------------------

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# Confirm expected columns exist
missing_target_columns = [
    t for t in TARGETS
    if t not in gold_all.columns
]

if missing_target_columns:
    raise ValueError(
        "These target columns were not found in train.csv:\n"
        f"{missing_target_columns}"
    )


# ------------------------------------------------------------
# 3. Extract the 58 genuinely labelled studies
# ------------------------------------------------------------

gold_mask = gold_all[TARGETS].notna().all(axis=1)

gold_df = gold_all.loc[
    gold_mask,
    ["StudyInstanceUID"] + TARGETS
].copy()

print("\n" + "=" * 80)
print("GOLD STUDIES")
print("=" * 80)

print("Gold rows:", len(gold_df))
print("Unique gold studies:", gold_df["StudyInstanceUID"].nunique())

if len(gold_df) != 58:
    raise ValueError(
        f"Expected 58 gold studies, found {len(gold_df)}"
    )


# ------------------------------------------------------------
# 4. Convert gold labels to same target-column names
# ------------------------------------------------------------

gold_rename = {
    target: f"{target}_target"
    for target in TARGETS
}

gold_df = gold_df.rename(columns=gold_rename)

gold_df["SupervisionSource"] = "Gold"
gold_df["PseudoSource"] = np.nan


# ------------------------------------------------------------
# 5. Prepare existing 4,349 pseudo studies
#
# IMPORTANT:
# Ignore the temporary PseudoBaseWeight from the previous cell.
# We are NOT setting final weights here.
# ------------------------------------------------------------

pseudo_final = pseudo_df.copy()

if "PseudoBaseWeight" in pseudo_final.columns:
    pseudo_final = pseudo_final.drop(
        columns=["PseudoBaseWeight"]
    )

pseudo_final["SupervisionSource"] = "Pseudo"


# ------------------------------------------------------------
# 6. Keep exactly the fields required
# ------------------------------------------------------------

target_cols = [
    f"{t}_target"
    for t in TARGETS
]

pseudo_final = pseudo_final[
    ["StudyInstanceUID",
     "SupervisionSource",
     "PseudoSource"] +
    target_cols
].copy()

gold_df = gold_df[
    ["StudyInstanceUID",
     "SupervisionSource",
     "PseudoSource"] +
    target_cols
].copy()


# ------------------------------------------------------------
# 7. Verify NO overlap
#
# Gold 58 should not appear in the 4,349 production set.
# ------------------------------------------------------------

overlap = set(gold_df["StudyInstanceUID"]).intersection(
    set(pseudo_final["StudyInstanceUID"])
)

print("\n" + "=" * 80)
print("GOLD / PSEUDO OVERLAP CHECK")
print("=" * 80)

print("Overlap:", len(overlap))

if len(overlap) != 0:
    raise ValueError(
        f"Gold and pseudo sets overlap by {len(overlap)} studies."
    )

print("✓ Gold and pseudo studies are completely separate")


# ------------------------------------------------------------
# 8. Combine into final 4,407-study table
# ------------------------------------------------------------

supervision_df = pd.concat(
    [gold_df, pseudo_final],
    axis=0,
    ignore_index=True
)

print("\n" + "=" * 80)
print("FINAL SUPERVISION TABLE")
print("=" * 80)

print("Shape:", supervision_df.shape)

print("\nSupervision source:")
print(
    supervision_df[
        "SupervisionSource"
    ].value_counts()
)

print("\nPseudo source:")
print(
    supervision_df.loc[
        supervision_df["SupervisionSource"] == "Pseudo",
        "PseudoSource"
    ].value_counts()
)

print("\nUnique studies:")
print(
    supervision_df[
        "StudyInstanceUID"
    ].nunique()
)

print("\nDuplicate studies:")
print(
    supervision_df[
        "StudyInstanceUID"
    ].duplicated().sum()
)

print("\nMissing target values:")
print(
    supervision_df[
        target_cols
    ].isna().sum().sum()
)


# ------------------------------------------------------------
# 9. Critical assertions
# ------------------------------------------------------------

assert len(supervision_df) == 4407
assert supervision_df["StudyInstanceUID"].nunique() == 4407
assert supervision_df["StudyInstanceUID"].duplicated().sum() == 0
assert supervision_df[target_cols].isna().sum().sum() == 0

assert (
    supervision_df["SupervisionSource"] == "Gold"
).sum() == 58

assert (
    supervision_df["SupervisionSource"] == "Pseudo"
).sum() == 4349


# ------------------------------------------------------------
# 10. Save intermediate training table
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/llm_soft_label_production"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUPERVISION_CSV = (
    OUTPUT_DIR /
    "final_4407_soft_supervision_table.csv"
)

supervision_df.to_csv(
    SUPERVISION_CSV,
    index=False
)

print("\nSaved:")
print(SUPERVISION_CSV)


# ------------------------------------------------------------
# 11. Preview
# ------------------------------------------------------------

display(
    supervision_df.head()
)

print("\n" + "=" * 80)
print("✓ 58 official gold studies")
print("✓ 4,349 continuous pseudo-labelled studies")
print("✓ 4,407 total studies")
print("✓ No hard pseudo-labels")
print("✓ No API calls")
print("✓ Final loss weights NOT assigned yet")
print("=" * 80)

In [ ]:
# ============================================================
# NEXT STEP — CONNECT SUPERVISION TABLE TO MRI SERIES
# CPU ONLY — NO API
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Load supervision table
# ------------------------------------------------------------

SUPERVISION_CSV = Path(
    "/kaggle/working/llm_soft_label_production/"
    "final_4407_soft_supervision_table.csv"
)

supervision_df = pd.read_csv(SUPERVISION_CSV)

print("=" * 80)
print("SUPERVISION TABLE")
print("=" * 80)

print("Rows:", len(supervision_df))
print(
    "Unique studies:",
    supervision_df["StudyInstanceUID"].nunique()
)


# ------------------------------------------------------------
# 2. Load official series metadata
# ------------------------------------------------------------

SERIES_CSV = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection/train_series.csv"
)

series_df = pd.read_csv(SERIES_CSV)

print("\n" + "=" * 80)
print("TRAIN SERIES")
print("=" * 80)

print("Shape:", series_df.shape)

print("\nColumns:")
for i, c in enumerate(series_df.columns, 1):
    print(f"{i:>2}. {c}")

print(
    "\nUnique studies in train_series:",
    series_df["StudyInstanceUID"].nunique()
)


# ------------------------------------------------------------
# 3. Keep only our 4,407 supervised studies
# ------------------------------------------------------------

supervised_series_df = series_df[
    series_df["StudyInstanceUID"].isin(
        supervision_df["StudyInstanceUID"]
    )
].copy()

print("\n" + "=" * 80)
print("SUPERVISED MRI SERIES")
print("=" * 80)

print("Series rows:", len(supervised_series_df))

print(
    "Unique supervised studies with MRI:",
    supervised_series_df["StudyInstanceUID"].nunique()
)


# ------------------------------------------------------------
# 4. Check whether every supervision study has MRI
# ------------------------------------------------------------

supervision_ids = set(
    supervision_df["StudyInstanceUID"]
)

mri_ids = set(
    supervised_series_df["StudyInstanceUID"]
)

missing_mri = supervision_ids - mri_ids

print("\nMissing MRI studies:", len(missing_mri))

if len(missing_mri) == 0:
    print("✓ All 4,407 supervision studies have MRI series")
else:
    print("⚠ Studies missing MRI:")
    print(list(missing_mri)[:20])


# ------------------------------------------------------------
# 5. MRI series counts by anatomical plane
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SERIES BY ANATOMICAL PLANE")
print("=" * 80)

print(
    supervised_series_df[
        "Anatomical_Plane"
    ].value_counts(dropna=False)
)


# ------------------------------------------------------------
# 6. Count number of series per study
# ------------------------------------------------------------

series_per_study = (
    supervised_series_df
    .groupby("StudyInstanceUID")
    .size()
)

print("\n" + "=" * 80)
print("SERIES PER STUDY")
print("=" * 80)

print(series_per_study.describe())


# ------------------------------------------------------------
# 7. Count available planes per study
# ------------------------------------------------------------

planes_per_study = (
    supervised_series_df
    .groupby("StudyInstanceUID")["Anatomical_Plane"]
    .nunique()
)

print("\n" + "=" * 80)
print("PLANES PER STUDY")
print("=" * 80)

print(
    planes_per_study.value_counts().sort_index()
)


# ------------------------------------------------------------
# 8. Gold vs pseudo MRI availability
# ------------------------------------------------------------

study_source = supervision_df[
    ["StudyInstanceUID", "SupervisionSource"]
]

series_with_source = supervised_series_df.merge(
    study_source,
    on="StudyInstanceUID",
    how="left"
)

print("\n" + "=" * 80)
print("MRI SERIES BY SUPERVISION SOURCE")
print("=" * 80)

print(
    series_with_source[
        "SupervisionSource"
    ].value_counts()
)

print("\nUnique studies by source:")

print(
    series_with_source[
        ["StudyInstanceUID", "SupervisionSource"]
    ]
    .drop_duplicates()
    ["SupervisionSource"]
    .value_counts()
)


# ------------------------------------------------------------
# 9. Final assertions
# ------------------------------------------------------------

assert len(supervision_df) == 4407
assert supervision_df["StudyInstanceUID"].nunique() == 4407
assert supervised_series_df["StudyInstanceUID"].nunique() == 4407

print("\n" + "=" * 80)
print("✓ Supervision ↔ MRI linkage complete")
print("✓ All 4,407 studies accounted for")
print("✓ No API calls")
print("✓ GPU still not needed")
print("=" * 80)

In [ ]:
# ============================================================
# FIND YESTERDAY'S SAVED SOFT-LABEL FILES
# FAST SEARCH — NO API
# ============================================================

from pathlib import Path

NOTEBOOK_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37"
)

print("=" * 80)
print("AVAILABLE NOTEBOOK INPUTS")
print("=" * 80)

for p in NOTEBOOK_ROOT.iterdir():
    if p.is_dir():
        print(p)


print("\n" + "=" * 80)
print("SOFT LABEL FILES")
print("=" * 80)

# Search ONLY our notebook inputs, not the MRI competition data
supervision_matches = list(
    NOTEBOOK_ROOT.glob(
        "*/llm_soft_label_production/"
        "final_4407_soft_supervision_table.csv"
    )
)

production_matches = list(
    NOTEBOOK_ROOT.glob(
        "*/llm_soft_label_production/"
        "production_predictions_snapshot.csv"
    )
)

print("\nSupervision table:")
for p in supervision_matches:
    print("✓", p)

print("\nProduction snapshot:")
for p in production_matches:
    print("✓", p)

print("\nCounts:")
print("Supervision matches:", len(supervision_matches))
print("Production matches:", len(production_matches))

In [ ]:
# ============================================================
# RESUME CELL — SOFT LABEL MRI PROJECT
# CPU ONLY — NO API
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("RESUMING RSNA KNEE MRI — SOFT LABEL PIPELINE")
print("=" * 80)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

COMP_ROOT = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

SAVED_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-soft-label-production/"
    "llm_soft_label_production"
)

SUPERVISION_CSV = (
    SAVED_ROOT /
    "final_4407_soft_supervision_table.csv"
)

PRODUCTION_CSV = (
    SAVED_ROOT /
    "production_predictions_snapshot.csv"
)

TRAIN_SERIES_CSV = (
    COMP_ROOT /
    "train_series.csv"
)

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

for p in [
    SUPERVISION_CSV,
    PRODUCTION_CSV,
    TRAIN_SERIES_CSV,
]:
    print(f"\n{p}")
    print("Exists:", p.exists())

# ------------------------------------------------------------
# Reload tables
# ------------------------------------------------------------

supervision_df = pd.read_csv(SUPERVISION_CSV)
production_df = pd.read_csv(PRODUCTION_CSV)
series_df = pd.read_csv(TRAIN_SERIES_CSV)

# ------------------------------------------------------------
# Rebuild MRI linkage
# ------------------------------------------------------------

supervised_series_df = series_df[
    series_df["StudyInstanceUID"].isin(
        supervision_df["StudyInstanceUID"]
    )
].copy()

study_source = supervision_df[
    ["StudyInstanceUID", "SupervisionSource"]
]

series_with_source = supervised_series_df.merge(
    study_source,
    on="StudyInstanceUID",
    how="left"
)

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESUME CHECK")
print("=" * 80)

print("Supervision rows:", len(supervision_df))
print(
    "Unique supervision studies:",
    supervision_df["StudyInstanceUID"].nunique()
)

print("\nSupervision source:")
print(
    supervision_df["SupervisionSource"].value_counts()
)

print("\nMRI series rows:", len(supervised_series_df))
print(
    "Unique MRI studies:",
    supervised_series_df["StudyInstanceUID"].nunique()
)

print("\nMRI series by source:")
print(
    series_with_source["SupervisionSource"].value_counts()
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(supervision_df) == 4407
assert supervision_df["StudyInstanceUID"].nunique() == 4407

assert (
    supervision_df["SupervisionSource"] == "Gold"
).sum() == 58

assert (
    supervision_df["SupervisionSource"] == "Pseudo"
).sum() == 4349

assert supervised_series_df["StudyInstanceUID"].nunique() == 4407

print("\n" + "=" * 80)
print("✓ RESUME SUCCESSFUL")
print("✓ 4,407 supervision studies restored")
print("✓ MRI linkage restored")
print("✓ No API calls")
print("✓ GPU still not needed")
print("=" * 80)

In [ ]:
# ============================================================
# NEXT STEP — PREFLIGHT EXISTING MRI ASSETS
# CPU ONLY — NO API
# ============================================================

from pathlib import Path
import pandas as pd

SSL_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-clean-ssl-training"
)

print("=" * 80)
print("MRI ASSET PREFLIGHT")
print("=" * 80)

print("\nSSL root:")
print(SSL_ROOT)
print("Exists:", SSL_ROOT.exists())

if not SSL_ROOT.exists():
    raise FileNotFoundError(
        f"Clean SSL training input not found:\n{SSL_ROOT}"
    )

# ------------------------------------------------------------
# Known reusable assets from our clean pipeline
# ------------------------------------------------------------

asset_candidates = {
    "encoder": SSL_ROOT / "ssl_training" / "best_encoder.pt",
    "percentiles": SSL_ROOT / "ssl_prepared" / "series_percentiles.parquet",
    "clean_triplets": SSL_ROOT / "ssl_prepared" / "ssl_clean_triplets.parquet",
    "series_split": SSL_ROOT / "ssl_prepared" / "series_split.csv",
    "study_split": SSL_ROOT / "ssl_prepared" / "study_split.csv",
}

print("\n" + "=" * 80)
print("EXPECTED ASSETS")
print("=" * 80)

for name, path in asset_candidates.items():
    print(f"\n{name}:")
    print(path)
    print("Exists:", path.exists())

# ------------------------------------------------------------
# Load triplet manifest if available
# ------------------------------------------------------------

TRIPLET_PATH = asset_candidates["clean_triplets"]

if TRIPLET_PATH.exists():
    triplets_df = pd.read_parquet(TRIPLET_PATH)

    print("\n" + "=" * 80)
    print("CLEAN TRIPLET MANIFEST")
    print("=" * 80)

    print("Shape:", triplets_df.shape)

    print("\nColumns:")
    for i, c in enumerate(triplets_df.columns, 1):
        print(f"{i:>2}. {c}")

    study_candidates = [
        c for c in triplets_df.columns
        if "study" in c.lower() and "uid" in c.lower()
    ]

    series_candidates = [
        c for c in triplets_df.columns
        if "series" in c.lower() and "uid" in c.lower()
    ]

    print("\nStudy UID candidates:")
    print(study_candidates)

    print("\nSeries UID candidates:")
    print(series_candidates)

    if study_candidates:
        study_col = study_candidates[0]

        print("\nUnique studies represented:")
        print(triplets_df[study_col].nunique())

        # Compare to our supervision table
        triplet_studies = set(
            triplets_df[study_col].astype(str)
        )

        gold_ids = set(
            supervision_df.loc[
                supervision_df["SupervisionSource"] == "Gold",
                "StudyInstanceUID"
            ].astype(str)
        )

        pseudo_ids = set(
            supervision_df.loc[
                supervision_df["SupervisionSource"] == "Pseudo",
                "StudyInstanceUID"
            ].astype(str)
        )

        print("\nPseudo studies represented:")
        print(len(pseudo_ids.intersection(triplet_studies)))

        print("Gold studies represented:")
        print(len(gold_ids.intersection(triplet_studies)))

else:
    print("\n⚠ ssl_clean_triplets.parquet not found")

# ------------------------------------------------------------
# Percentile table check
# ------------------------------------------------------------

PERCENTILE_PATH = asset_candidates["percentiles"]

if PERCENTILE_PATH.exists():
    percentile_df = pd.read_parquet(PERCENTILE_PATH)

    print("\n" + "=" * 80)
    print("SERIES PERCENTILES")
    print("=" * 80)

    print("Shape:", percentile_df.shape)

    print("\nColumns:")
    for i, c in enumerate(percentile_df.columns, 1):
        print(f"{i:>2}. {c}")

# ------------------------------------------------------------
# Encoder check
# ------------------------------------------------------------

ENCODER_PATH = asset_candidates["encoder"]

print("\n" + "=" * 80)
print("ENCODER")
print("=" * 80)

print("Path:", ENCODER_PATH)
print("Exists:", ENCODER_PATH.exists())

print("\n" + "=" * 80)
print("✓ Preflight complete")
print("✓ No API calls")
print("✓ GPU still not needed")
print("=" * 80)

In [ ]:
# ============================================================
# STEP — PREPARE GOLD-ONLY MRI SERIES INVENTORY
# CPU ONLY — NO API
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 80)
print("GOLD MRI SERIES INVENTORY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Gold study IDs
# ------------------------------------------------------------

gold_ids = set(
    supervision_df.loc[
        supervision_df["SupervisionSource"] == "Gold",
        "StudyInstanceUID"
    ].astype(str)
)

print("\nGold studies:", len(gold_ids))

if len(gold_ids) != 58:
    raise ValueError(
        f"Expected 58 gold studies, found {len(gold_ids)}"
    )

# ------------------------------------------------------------
# 2. Gold MRI series
# ------------------------------------------------------------

gold_series_df = series_df[
    series_df["StudyInstanceUID"].astype(str).isin(gold_ids)
].copy()

print("\nGold MRI series:", len(gold_series_df))
print(
    "Unique gold studies:",
    gold_series_df["StudyInstanceUID"].nunique()
)

# ------------------------------------------------------------
# 3. Plane distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GOLD SERIES BY PLANE")
print("=" * 80)

print(
    gold_series_df[
        "Anatomical_Plane"
    ].value_counts()
)

# ------------------------------------------------------------
# 4. Series per study
# ------------------------------------------------------------

series_per_gold_study = (
    gold_series_df
    .groupby("StudyInstanceUID")
    .size()
)

print("\n" + "=" * 80)
print("SERIES PER GOLD STUDY")
print("=" * 80)

print(series_per_gold_study.describe())

# ------------------------------------------------------------
# 5. Plane coverage
# ------------------------------------------------------------

planes_per_gold_study = (
    gold_series_df
    .groupby("StudyInstanceUID")["Anatomical_Plane"]
    .nunique()
)

print("\n" + "=" * 80)
print("PLANES PER GOLD STUDY")
print("=" * 80)

print(
    planes_per_gold_study
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 6. Check overlap with existing pseudo manifest
# ------------------------------------------------------------

pseudo_manifest_ids = set(
    triplets_df["StudyInstanceUID"].astype(str)
)

overlap = gold_ids.intersection(pseudo_manifest_ids)

print("\n" + "=" * 80)
print("OVERLAP WITH EXISTING PSEUDO TRIPLETS")
print("=" * 80)

print("Gold studies already in pseudo manifest:", len(overlap))

if len(overlap) == 0:
    print("✓ Correct — gold studies are completely separate")
else:
    print("⚠ Unexpected overlap found")

# ------------------------------------------------------------
# 7. Save gold inventory
# ------------------------------------------------------------

GOLD_WORK = Path(
    "/kaggle/working/gold_supervised_mri"
)

GOLD_WORK.mkdir(
    parents=True,
    exist_ok=True
)

GOLD_SERIES_CSV = (
    GOLD_WORK /
    "gold_58_series_inventory.csv"
)

gold_series_df.to_csv(
    GOLD_SERIES_CSV,
    index=False
)

print("\nSaved:")
print(GOLD_SERIES_CSV)

# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert len(gold_ids) == 58
assert gold_series_df["StudyInstanceUID"].nunique() == 58
assert len(gold_series_df) == 336
assert len(overlap) == 0

print("\n" + "=" * 80)
print("✓ 58 gold studies identified")
print("✓ 336 gold MRI series identified")
print("✓ Gold studies separated from pseudo manifest")
print("✓ No API calls")
print("✓ GPU still not needed")
print("=" * 80)

In [ ]:
# ============================================================
# CHECK ACTUAL TRAIN IMAGE FOLDER STRUCTURE
# CPU ONLY — NO API
# ============================================================

from pathlib import Path

COMP_ROOT = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

print("=" * 80)
print("COMPETITION IMAGE PATH CHECK")
print("=" * 80)

print("\nCompetition root:")
print(COMP_ROOT)

print("\nTop-level folders/files:")
for p in sorted(COMP_ROOT.iterdir()):
    print(" ", p.name)

# Look for likely image directories
image_dirs = [
    p for p in COMP_ROOT.iterdir()
    if p.is_dir()
]

print("\nDirectories:")
for p in image_dirs:
    print(" ", p)

# ------------------------------------------------------------
# Pick one known gold study + series
# ------------------------------------------------------------

sample = gold_series_df.iloc[0]

study_uid = str(sample["StudyInstanceUID"])
series_uid = str(sample["SeriesInstanceUID"])

print("\n" + "=" * 80)
print("SAMPLE GOLD SERIES")
print("=" * 80)

print("Study:", study_uid)
print("Series:", series_uid)

# Test common layouts
candidates = [
    COMP_ROOT / "train_images" / study_uid / series_uid,
    COMP_ROOT / "train_images" / series_uid,
    COMP_ROOT / "train" / study_uid / series_uid,
    COMP_ROOT / "train" / series_uid,
    COMP_ROOT / study_uid / series_uid,
    COMP_ROOT / series_uid,
]

print("\nCandidate paths:")

for p in candidates:
    print(f"{p}")
    print("  Exists:", p.exists())

    if p.exists() and p.is_dir():
        files = list(p.iterdir())
        print("  Files:", len(files))
        print("  First few:", [x.name for x in files[:5]])

print("\n" + "=" * 80)
print("✓ Diagnostic complete")
print("✓ No API calls")
print("✓ GPU not needed")
print("=" * 80)

In [ ]:
# ============================================================
# CONFIRM ACTUAL TRAIN_SERIES FOLDER STRUCTURE
# CPU ONLY — NO API
# ============================================================

from pathlib import Path

TRAIN_SERIES_ROOT = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection/train_series"
)

sample = gold_series_df.iloc[0]

study_uid = str(sample["StudyInstanceUID"])
series_uid = str(sample["SeriesInstanceUID"])

print("=" * 80)
print("TRAIN_SERIES STRUCTURE CHECK")
print("=" * 80)

print("\nRoot exists:", TRAIN_SERIES_ROOT.exists())

candidates = [
    TRAIN_SERIES_ROOT / study_uid / series_uid,
    TRAIN_SERIES_ROOT / series_uid,
]

for p in candidates:
    print("\nCandidate:")
    print(p)
    print("Exists:", p.exists())

    if p.exists():
        files = list(p.iterdir())
        print("Files:", len(files))
        print("First 5:")
        for f in files[:5]:
            print(" ", f.name)

print("\nTop-level sample folders:")
for p in list(TRAIN_SERIES_ROOT.iterdir())[:5]:
    print(" ", p.name)

In [ ]:
# ============================================================
# GOLD 58 — CORRECTED SLICE INDEX + 2.5D TRIPLETS
# CPU ONLY — NO API
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import pydicom
from tqdm.auto import tqdm

print("=" * 80)
print("BUILDING GOLD 2.5D TRIPLETS — CORRECTED PATH")
print("=" * 80)

TRAIN_SERIES_ROOT = Path(
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection/train_series"
)

GOLD_WORK = Path(
    "/kaggle/working/gold_supervised_mri"
)

GOLD_WORK.mkdir(
    parents=True,
    exist_ok=True
)

GOLD_SLICE_INDEX = (
    GOLD_WORK /
    "gold_58_slice_index.parquet"
)

GOLD_TRIPLETS = (
    GOLD_WORK /
    "gold_58_triplets.parquet"
)

print("\nTrain series root exists:", TRAIN_SERIES_ROOT.exists())


# ------------------------------------------------------------
# Helper — slice position
# ------------------------------------------------------------

def get_slice_position(ds):

    if hasattr(ds, "ImagePositionPatient"):
        try:
            ipp = np.asarray(
                ds.ImagePositionPatient,
                dtype=float
            )

            if len(ipp) == 3:
                return float(ipp[2]), "ImagePositionPatient"

        except Exception:
            pass

    if hasattr(ds, "SliceLocation"):
        try:
            return float(ds.SliceLocation), "SliceLocation"
        except Exception:
            pass

    if hasattr(ds, "InstanceNumber"):
        try:
            return float(ds.InstanceNumber), "InstanceNumber"
        except Exception:
            pass

    return np.nan, "Unknown"


# ------------------------------------------------------------
# 1. Build slice index
# ------------------------------------------------------------

slice_rows = []
failed_files = []
missing_series = []

for row in tqdm(
    gold_series_df.itertuples(index=False),
    total=len(gold_series_df),
    desc="Indexing gold MRI series"
):

    study_uid = str(row.StudyInstanceUID)
    series_uid = str(row.SeriesInstanceUID)

    series_path = (
        TRAIN_SERIES_ROOT /
        study_uid /
        series_uid
    )

    if not series_path.exists():
        missing_series.append(str(series_path))
        continue

    dcm_files = list(
        series_path.glob("*.dcm")
    )

    for dcm_path in dcm_files:

        try:

            ds = pydicom.dcmread(
                dcm_path,
                stop_before_pixels=True,
                force=True
            )

            position, position_source = (
                get_slice_position(ds)
            )

            instance_number = (
                int(ds.InstanceNumber)
                if hasattr(ds, "InstanceNumber")
                else np.nan
            )

            rows = (
                int(ds.Rows)
                if hasattr(ds, "Rows")
                else np.nan
            )

            cols = (
                int(ds.Columns)
                if hasattr(ds, "Columns")
                else np.nan
            )

            spacing_y = np.nan
            spacing_x = np.nan

            if hasattr(ds, "PixelSpacing"):
                try:
                    spacing_y = float(ds.PixelSpacing[0])
                    spacing_x = float(ds.PixelSpacing[1])
                except Exception:
                    pass

            slice_rows.append({
                "StudyInstanceUID": study_uid,
                "SeriesInstanceUID": series_uid,
                "Anatomical_Plane": row.Anatomical_Plane,
                "Path": str(dcm_path),
                "Position": position,
                "PositionSource": position_source,
                "InstanceNumber": instance_number,
                "Rows": rows,
                "Columns": cols,
                "PixelSpacingY": spacing_y,
                "PixelSpacingX": spacing_x,
            })

        except Exception as e:

            failed_files.append({
                "Path": str(dcm_path),
                "Error": str(e),
            })


gold_slice_df = pd.DataFrame(slice_rows)

print("\n" + "=" * 80)
print("GOLD SLICE INDEX")
print("=" * 80)

print("Indexed slices:", len(gold_slice_df))

if len(gold_slice_df) == 0:
    raise RuntimeError(
        "No gold slices were indexed. Stop here."
    )

print(
    "Unique studies:",
    gold_slice_df["StudyInstanceUID"].nunique()
)

print(
    "Unique series:",
    gold_slice_df["SeriesInstanceUID"].nunique()
)

print("Missing series folders:", len(missing_series))
print("Failed DICOM headers:", len(failed_files))


# ------------------------------------------------------------
# 2. Sort slices within each series
# ------------------------------------------------------------

sorted_groups = []

for _, group in gold_slice_df.groupby(
    ["StudyInstanceUID", "SeriesInstanceUID"],
    sort=False
):

    group = group.copy()

    if group["Position"].notna().all():
        group = group.sort_values(
            ["Position", "InstanceNumber"],
            kind="mergesort"
        )
    else:
        group = group.sort_values(
            "InstanceNumber",
            kind="mergesort"
        )

    sorted_groups.append(
        group.reset_index(drop=True)
    )

gold_slice_df = pd.concat(
    sorted_groups,
    ignore_index=True
)


# ------------------------------------------------------------
# 3. Save slice index
# ------------------------------------------------------------

gold_slice_df.to_parquet(
    GOLD_SLICE_INDEX,
    index=False
)

print("\nSaved slice index:")
print(GOLD_SLICE_INDEX)


# ------------------------------------------------------------
# 4. Build previous / centre / next triplets
# ------------------------------------------------------------

triplet_rows = []

series_groups = gold_slice_df.groupby(
    ["StudyInstanceUID", "SeriesInstanceUID"],
    sort=False
)

for (study_uid, series_uid), group in tqdm(
    series_groups,
    total=gold_slice_df[
        ["StudyInstanceUID", "SeriesInstanceUID"]
    ].drop_duplicates().shape[0],
    desc="Building gold triplets"
):

    group = group.reset_index(drop=True)

    n = len(group)

    if n < 3:
        continue

    for centre_idx in range(1, n - 1):

        prev_row = group.iloc[centre_idx - 1]
        centre_row = group.iloc[centre_idx]
        next_row = group.iloc[centre_idx + 1]

        triplet_rows.append({
            "StudyInstanceUID": study_uid,
            "SeriesInstanceUID": series_uid,
            "Anatomical_Plane": centre_row["Anatomical_Plane"],

            "CentrePath": centre_row["Path"],
            "CentrePosition": centre_row["Position"],
            "PositionSource": centre_row["PositionSource"],
            "InstanceNumber": centre_row["InstanceNumber"],
            "Rows": centre_row["Rows"],
            "Columns": centre_row["Columns"],
            "PixelSpacingY": centre_row["PixelSpacingY"],
            "PixelSpacingX": centre_row["PixelSpacingX"],
            "CentreIndex": centre_idx,

            "PreviousPath": prev_row["Path"],
            "NextPath": next_row["Path"],
            "PreviousPosition": prev_row["Position"],
            "NextPosition": next_row["Position"],

            "SupervisionSource": "Gold",
        })

gold_triplets_df = pd.DataFrame(triplet_rows)


# ------------------------------------------------------------
# 5. QC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GOLD TRIPLET QC")
print("=" * 80)

print("Triplets:", len(gold_triplets_df))

print(
    "Unique studies:",
    gold_triplets_df["StudyInstanceUID"].nunique()
)

print(
    "Unique series:",
    gold_triplets_df["SeriesInstanceUID"].nunique()
)

print("\nTriplets by plane:")
print(
    gold_triplets_df[
        "Anatomical_Plane"
    ].value_counts()
)

print("\nTriplets per series:")
print(
    gold_triplets_df
    .groupby("SeriesInstanceUID")
    .size()
    .describe()
)


# ------------------------------------------------------------
# 6. Critical checks
# ------------------------------------------------------------

assert gold_triplets_df["StudyInstanceUID"].nunique() == 58
assert gold_triplets_df["SeriesInstanceUID"].nunique() == 336

assert gold_triplets_df[
    ["PreviousPath", "CentrePath", "NextPath"]
].isna().sum().sum() == 0


# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

gold_triplets_df.to_parquet(
    GOLD_TRIPLETS,
    index=False
)

print("\nSaved gold triplets:")
print(GOLD_TRIPLETS)


# ------------------------------------------------------------
# 8. Schema comparison with pseudo triplets
# ------------------------------------------------------------

pseudo_core = [
    c for c in triplets_df.columns
    if c != "SSL_Split"
]

gold_core = [
    c for c in gold_triplets_df.columns
    if c != "SupervisionSource"
]

print("\n" + "=" * 80)
print("SCHEMA COMPARISON")
print("=" * 80)

print("Pseudo core columns:", len(pseudo_core))
print("Gold core columns:", len(gold_core))
print("Core schemas match:", pseudo_core == gold_core)

print("\n" + "=" * 80)
print("✓ Gold slice indexing complete")
print("✓ Gold 2.5D triplets built")
print("✓ Existing 4,349 pseudo triplets untouched")
print("✓ No API calls")
print("✓ No SSL retraining")
print("✓ GPU still not needed")
print("=" * 80)

In [ ]:
# ============================================================
# GOLD 58 — COMPUTE PER-SERIES P1 / P99
# CPU ONLY — NO API
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import pydicom
from tqdm.auto import tqdm

print("=" * 80)
print("GOLD SERIES INTENSITY PERCENTILES")
print("=" * 80)

GOLD_WORK = Path(
    "/kaggle/working/gold_supervised_mri"
)

GOLD_PERCENTILES = (
    GOLD_WORK /
    "gold_58_series_percentiles.parquet"
)

# ------------------------------------------------------------
# 1. Unique gold series
# ------------------------------------------------------------

gold_series_unique = (
    gold_slice_df[
        [
            "StudyInstanceUID",
            "SeriesInstanceUID"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nGold series:", len(gold_series_unique))

# ------------------------------------------------------------
# 2. Compute percentiles
# ------------------------------------------------------------

percentile_rows = []

for row in tqdm(
    gold_series_unique.itertuples(index=False),
    total=len(gold_series_unique),
    desc="Computing gold percentiles"
):

    study_uid = str(row.StudyInstanceUID)
    series_uid = str(row.SeriesInstanceUID)

    series_slices = gold_slice_df[
        gold_slice_df["SeriesInstanceUID"] == series_uid
    ]

    sampled_pixels = []

    valid = True

    for path in series_slices["Path"]:

        try:
            ds = pydicom.dcmread(
                path,
                force=True
            )

            arr = ds.pixel_array.astype(
                np.float32
            )

            # Apply rescale if present
            slope = float(
                getattr(ds, "RescaleSlope", 1.0)
            )

            intercept = float(
                getattr(ds, "RescaleIntercept", 0.0)
            )

            arr = arr * slope + intercept

            # Downsample pixels for percentile estimation
            # to keep CPU/RAM use reasonable
            flat = arr.ravel()

            if len(flat) > 50000:
                idx = np.linspace(
                    0,
                    len(flat) - 1,
                    50000,
                    dtype=int
                )
                flat = flat[idx]

            sampled_pixels.append(flat)

        except Exception:
            valid = False
            continue

    if len(sampled_pixels) == 0:

        percentile_rows.append({
            "SeriesInstanceUID": series_uid,
            "P1": np.nan,
            "P99": np.nan,
            "NumSlices": len(series_slices),
            "Valid": False,
        })

        continue

    pixels = np.concatenate(
        sampled_pixels
    )

    p1 = float(
        np.percentile(pixels, 1)
    )

    p99 = float(
        np.percentile(pixels, 99)
    )

    if (
        not np.isfinite(p1)
        or not np.isfinite(p99)
        or p99 <= p1
    ):
        valid = False

    percentile_rows.append({
        "SeriesInstanceUID": series_uid,
        "P1": p1,
        "P99": p99,
        "NumSlices": len(series_slices),
        "Valid": valid,
    })


gold_percentile_df = pd.DataFrame(
    percentile_rows
)

# ------------------------------------------------------------
# 3. QC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GOLD PERCENTILE QC")
print("=" * 80)

print("Rows:", len(gold_percentile_df))

print(
    "\nValid counts:"
)

print(
    gold_percentile_df[
        "Valid"
    ].value_counts(dropna=False)
)

print(
    "\nMissing P1:",
    gold_percentile_df["P1"].isna().sum()
)

print(
    "Missing P99:",
    gold_percentile_df["P99"].isna().sum()
)

print(
    "\nP1 summary:"
)

print(
    gold_percentile_df["P1"].describe()
)

print(
    "\nP99 summary:"
)

print(
    gold_percentile_df["P99"].describe()
)

print(
    "\nNumSlices summary:"
)

print(
    gold_percentile_df["NumSlices"].describe()
)

# ------------------------------------------------------------
# 4. Critical checks
# ------------------------------------------------------------

assert len(gold_percentile_df) == 336

assert (
    gold_percentile_df[
        "SeriesInstanceUID"
    ].nunique()
    == 336
)

assert (
    gold_percentile_df["P1"].isna().sum()
    == 0
)

assert (
    gold_percentile_df["P99"].isna().sum()
    == 0
)

# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------

gold_percentile_df.to_parquet(
    GOLD_PERCENTILES,
    index=False
)

print("\nSaved:")
print(GOLD_PERCENTILES)

print("\n" + "=" * 80)
print("✓ Gold P1/P99 normalization statistics complete")
print("✓ 336 gold series processed")
print("✓ No API calls")
print("✓ GPU still not needed")
print("=" * 80)

In [ ]:
# ============================================================
# BUILD COMBINED SUPERVISED MRI MANIFEST
# 58 GOLD + 4,349 PSEUDO
# CPU ONLY — NO API
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("BUILDING COMBINED SUPERVISED MRI MANIFEST")
print("=" * 80)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

CPU_OUT = Path(
    "/kaggle/working/soft_supervised_mri_cpu"
)

CPU_OUT.mkdir(
    parents=True,
    exist_ok=True
)

# Existing pseudo assets
PSEUDO_TRIPLETS = (
    SSL_ROOT /
    "ssl_prepared" /
    "ssl_clean_triplets.parquet"
)

PSEUDO_PERCENTILES = (
    SSL_ROOT /
    "ssl_prepared" /
    "series_percentiles.parquet"
)

# Newly built gold assets
GOLD_TRIPLETS = Path(
    "/kaggle/working/gold_supervised_mri/"
    "gold_58_triplets.parquet"
)

GOLD_PERCENTILES = Path(
    "/kaggle/working/gold_supervised_mri/"
    "gold_58_series_percentiles.parquet"
)

# ------------------------------------------------------------
# 2. Load
# ------------------------------------------------------------

pseudo_triplets = pd.read_parquet(
    PSEUDO_TRIPLETS
)

pseudo_percentiles = pd.read_parquet(
    PSEUDO_PERCENTILES
)

gold_triplets = pd.read_parquet(
    GOLD_TRIPLETS
)

gold_percentiles = pd.read_parquet(
    GOLD_PERCENTILES
)

print("\nPseudo triplets:", len(pseudo_triplets))
print("Gold triplets:", len(gold_triplets))

# ------------------------------------------------------------
# 3. Prepare pseudo triplets
# ------------------------------------------------------------

pseudo_triplets = pseudo_triplets.copy()

if "SSL_Split" in pseudo_triplets.columns:
    pseudo_triplets = pseudo_triplets.drop(
        columns=["SSL_Split"]
    )

pseudo_triplets["SupervisionSource"] = "Pseudo"

# ------------------------------------------------------------
# 4. Prepare gold triplets
# ------------------------------------------------------------

gold_triplets = gold_triplets.copy()

# Already marked Gold, but enforce it
gold_triplets["SupervisionSource"] = "Gold"

# ------------------------------------------------------------
# 5. Verify schemas before concat
# ------------------------------------------------------------

common_cols = [
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "Anatomical_Plane",
    "CentrePath",
    "CentrePosition",
    "PositionSource",
    "InstanceNumber",
    "Rows",
    "Columns",
    "PixelSpacingY",
    "PixelSpacingX",
    "CentreIndex",
    "PreviousPath",
    "NextPath",
    "PreviousPosition",
    "NextPosition",
    "SupervisionSource",
]

missing_pseudo = [
    c for c in common_cols
    if c not in pseudo_triplets.columns
]

missing_gold = [
    c for c in common_cols
    if c not in gold_triplets.columns
]

if missing_pseudo:
    raise ValueError(
        f"Pseudo triplets missing columns: {missing_pseudo}"
    )

if missing_gold:
    raise ValueError(
        f"Gold triplets missing columns: {missing_gold}"
    )

pseudo_triplets = pseudo_triplets[
    common_cols
].copy()

gold_triplets = gold_triplets[
    common_cols
].copy()

# ------------------------------------------------------------
# 6. Combine triplets
# ------------------------------------------------------------

combined_triplets = pd.concat(
    [
        pseudo_triplets,
        gold_triplets
    ],
    ignore_index=True
)

print("\n" + "=" * 80)
print("COMBINED TRIPLETS")
print("=" * 80)

print("Rows:", len(combined_triplets))

print(
    "Unique studies:",
    combined_triplets[
        "StudyInstanceUID"
    ].nunique()
)

print(
    "Unique series:",
    combined_triplets[
        "SeriesInstanceUID"
    ].nunique()
)

print("\nTriplets by source:")
print(
    combined_triplets[
        "SupervisionSource"
    ].value_counts()
)

# ------------------------------------------------------------
# 7. Combine percentile tables
# ------------------------------------------------------------

pseudo_percentiles = (
    pseudo_percentiles.copy()
)

gold_percentiles = (
    gold_percentiles.copy()
)

all_percentiles = pd.concat(
    [
        pseudo_percentiles,
        gold_percentiles
    ],
    ignore_index=True
)

print("\n" + "=" * 80)
print("COMBINED SERIES PERCENTILES")
print("=" * 80)

print("Rows:", len(all_percentiles))
print(
    "Unique series:",
    all_percentiles[
        "SeriesInstanceUID"
    ].nunique()
)

print(
    "Invalid series:",
    (~all_percentiles["Valid"]).sum()
)

# ------------------------------------------------------------
# 8. Merge percentiles onto triplets
# ------------------------------------------------------------

combined_manifest = (
    combined_triplets
    .merge(
        all_percentiles[
            [
                "SeriesInstanceUID",
                "P1",
                "P99",
                "NumSlices",
                "Valid",
            ]
        ],
        on="SeriesInstanceUID",
        how="left",
        validate="many_to_one"
    )
)

print("\nMissing percentile matches:")
print(
    combined_manifest[
        ["P1", "P99"]
    ].isna().sum()
)

# ------------------------------------------------------------
# 9. Attach study-level supervision targets
# ------------------------------------------------------------

target_cols = [
    c for c in supervision_df.columns
    if c.endswith("_target")
]

study_supervision = supervision_df[
    [
        "StudyInstanceUID",
        "SupervisionSource",
        "PseudoSource",
    ] + target_cols
].copy()

# Remove manifest source before merge to avoid duplicate naming
combined_manifest = (
    combined_manifest
    .drop(columns=["SupervisionSource"])
    .merge(
        study_supervision,
        on="StudyInstanceUID",
        how="left",
        validate="many_to_one"
    )
)

# ------------------------------------------------------------
# 10. QC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SUPERVISED MANIFEST QC")
print("=" * 80)

print("Rows:", len(combined_manifest))

print(
    "Unique studies:",
    combined_manifest[
        "StudyInstanceUID"
    ].nunique()
)

print(
    "Unique series:",
    combined_manifest[
        "SeriesInstanceUID"
    ].nunique()
)

print("\nSupervision source:")
print(
    combined_manifest[
        "SupervisionSource"
    ].value_counts()
)

print("\nPseudo source:")
print(
    combined_manifest.loc[
        combined_manifest["SupervisionSource"] == "Pseudo",
        "PseudoSource"
    ].value_counts()
)

print("\nMissing targets:")
print(
    combined_manifest[
        target_cols
    ].isna().sum().sum()
)

print("\nMissing P1/P99:")
print(
    combined_manifest[
        ["P1", "P99"]
    ].isna().sum()
)

# ------------------------------------------------------------
# 11. Critical assertions
# ------------------------------------------------------------

assert (
    combined_manifest[
        "StudyInstanceUID"
    ].nunique()
    == 4407
)

assert (
    combined_manifest[
        "SeriesInstanceUID"
    ].nunique()
    == 24371
)

assert (
    combined_manifest[
        target_cols
    ].isna().sum().sum()
    == 0
)

assert (
    combined_manifest[
        ["P1", "P99"]
    ].isna().sum().sum()
    == 0
)

assert (
    combined_manifest[
        "Valid"
    ].all()
)

# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

MANIFEST_OUT = (
    CPU_OUT /
    "combined_4407_supervised_triplets.parquet"
)

PERCENTILES_OUT = (
    CPU_OUT /
    "combined_24371_series_percentiles.parquet"
)

combined_manifest.to_parquet(
    MANIFEST_OUT,
    index=False
)

all_percentiles.to_parquet(
    PERCENTILES_OUT,
    index=False
)

print("\nSaved manifest:")
print(MANIFEST_OUT)

print("\nSaved percentiles:")
print(PERCENTILES_OUT)

print("\n" + "=" * 80)
print("✓ Combined supervised MRI manifest built")
print("✓ 4,407 studies")
print("✓ 24,371 series")
print("✓ Gold + soft pseudo targets attached")
print("✓ P1/P99 attached")
print("✓ No hard pseudo-labels")
print("✓ No API calls")
print("✓ GPU still not needed")
print("=" * 80)

In [ ]:
# ============================================================
# SAVE TRAINING CONFIG + SUPERVISION WEIGHT METADATA
# CPU ONLY — NO TRAINING
# ============================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 80)
print("FINAL TRAINING CONFIG PREPARATION")
print("=" * 80)

CPU_OUT = Path(
    "/kaggle/working/soft_supervised_mri_cpu"
)

# ------------------------------------------------------------
# 1. Target order — lock this permanently
# ------------------------------------------------------------

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

target_cols = [
    f"{t}_target"
    for t in TARGETS
]

# ------------------------------------------------------------
# 2. Training strategy metadata
#
# IMPORTANT:
# These define the intended GPU-stage objective.
# We are NOT training here.
# ------------------------------------------------------------

training_config = {

    "strategy": "gold_plus_soft_pseudo_supervision",

    "num_targets": 12,

    "targets": TARGETS,

    "pseudo_labels": {
        "type": "continuous_probabilities",
        "hard_thresholding": False,
        "primary_source": "Luna",
        "legacy_terra_rows_retained": True,
    },

    "supervision": {
        "gold": {
            "target_type": "official_binary_labels",
            "relative_weight": 1.0
        },

        "pseudo": {
            "target_type": "continuous_soft_labels",

            # Final absolute scale will be tuned in GPU CV.
            # This is our starting plan.
            "relative_weight": 0.25
        }
    },

    # Target-specific multiplier applied ONLY
    # to pseudo-supervision loss.
    "pseudo_target_multipliers": {
        "ACL": 1.0,
        "MCL": 1.0,
        "Medial Meniscus": 1.0,
        "Lateral Meniscus": 1.0,
        "Medial OA": 1.0,
        "Lateral OA": 1.0,
        "PF OA": 1.0,

        # Reduced trust
        "Effusion": 0.5,

        # Essentially excluded from pseudo supervision
        "Synovitis": 0.0,

        "Baker's": 1.0,
        "Contusion": 1.0,
        "Fracture": 1.0,
    },

    "initialization": {
        "use_pretrained_encoder": True,
        "encoder_file": "best_encoder.pt",
    },

    "validation": {
        "primary": "gold_only_cross_validation",
        "pseudo_rows_in_validation": False,
    },

    "gpu_training_notebook": True,
}

# ------------------------------------------------------------
# 3. Save JSON
# ------------------------------------------------------------

CONFIG_PATH = (
    CPU_OUT /
    "training_config.json"
)

with open(
    CONFIG_PATH,
    "w"
) as f:
    json.dump(
        training_config,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 4. Save compact target-weight table too
# ------------------------------------------------------------

weight_rows = []

for target in TARGETS:

    weight_rows.append({
        "Target": target,
        "GoldWeight": 1.0,
        "PseudoBaseWeight": 0.25,
        "PseudoTargetMultiplier":
            training_config[
                "pseudo_target_multipliers"
            ][target],
        "EffectivePseudoWeight":
            0.25 *
            training_config[
                "pseudo_target_multipliers"
            ][target],
    })

weights_df = pd.DataFrame(
    weight_rows
)

WEIGHTS_PATH = (
    CPU_OUT /
    "target_supervision_weights.csv"
)

weights_df.to_csv(
    WEIGHTS_PATH,
    index=False
)

# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

print("\nTraining config:")
print(json.dumps(
    training_config,
    indent=2
))

print("\nTarget weight table:")
display(weights_df)

print("\nSaved:")
print(CONFIG_PATH)
print(WEIGHTS_PATH)

print("\n" + "=" * 80)
print("✓ Training strategy metadata saved")
print("✓ Gold weight = 1.00")
print("✓ Pseudo starting weight = 0.25")
print("✓ Effusion pseudo multiplier = 0.50")
print("✓ Synovitis pseudo multiplier = 0.00")
print("✓ Continuous soft targets preserved")
print("✓ No training performed")
print("✓ GPU still OFF")
print("=" * 80)

In [ ]:
# ============================================================
# FINAL CPU INTEGRITY CHECK
# BEFORE MOVING TO NEW GPU NOTEBOOK
# ============================================================

from pathlib import Path
import pandas as pd
import json

print("=" * 80)
print("FINAL CPU PIPELINE INTEGRITY CHECK")
print("=" * 80)

CPU_OUT = Path(
    "/kaggle/working/soft_supervised_mri_cpu"
)

MANIFEST_PATH = (
    CPU_OUT /
    "combined_4407_supervised_triplets.parquet"
)

PERCENTILES_PATH = (
    CPU_OUT /
    "combined_24371_series_percentiles.parquet"
)

CONFIG_PATH = (
    CPU_OUT /
    "training_config.json"
)

WEIGHTS_PATH = (
    CPU_OUT /
    "target_supervision_weights.csv"
)

# ------------------------------------------------------------
# 1. Confirm required files
# ------------------------------------------------------------

required_files = [
    MANIFEST_PATH,
    PERCENTILES_PATH,
    CONFIG_PATH,
    WEIGHTS_PATH,
]

print("\nRequired files:")

for p in required_files:
    print(f"{p.name:<45} Exists: {p.exists()}")

    if not p.exists():
        raise FileNotFoundError(p)

# ------------------------------------------------------------
# 2. Reload FROM DISK
#
# Important: this proves the saved files work independently
# of variables currently living in notebook memory.
# ------------------------------------------------------------

manifest_check = pd.read_parquet(
    MANIFEST_PATH
)

percentiles_check = pd.read_parquet(
    PERCENTILES_PATH
)

weights_check = pd.read_csv(
    WEIGHTS_PATH
)

with open(CONFIG_PATH, "r") as f:
    config_check = json.load(f)

# ------------------------------------------------------------
# 3. Basic dimensions
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVED DATASET")
print("=" * 80)

print("Manifest rows:", len(manifest_check))

print(
    "Unique studies:",
    manifest_check["StudyInstanceUID"].nunique()
)

print(
    "Unique series:",
    manifest_check["SeriesInstanceUID"].nunique()
)

print(
    "Percentile rows:",
    len(percentiles_check)
)

# ------------------------------------------------------------
# 4. Supervision checks
# ------------------------------------------------------------

print("\nTriplets by supervision source:")

print(
    manifest_check[
        "SupervisionSource"
    ].value_counts()
)

study_summary = (
    manifest_check[
        [
            "StudyInstanceUID",
            "SupervisionSource",
            "PseudoSource",
        ]
    ]
    .drop_duplicates(
        subset=["StudyInstanceUID"]
    )
    .reset_index(drop=True)
)

print("\nStudies by supervision source:")

print(
    study_summary[
        "SupervisionSource"
    ].value_counts()
)

print("\nPseudo studies by source:")

print(
    study_summary.loc[
        study_summary["SupervisionSource"] == "Pseudo",
        "PseudoSource"
    ].value_counts()
)

# ------------------------------------------------------------
# 5. Target integrity
# ------------------------------------------------------------

TARGETS = config_check["targets"]

target_cols = [
    f"{t}_target"
    for t in TARGETS
]

print("\n" + "=" * 80)
print("TARGET INTEGRITY")
print("=" * 80)

print(
    "Missing target values:",
    manifest_check[
        target_cols
    ].isna().sum().sum()
)

for col in target_cols:

    mn = manifest_check[col].min()
    mx = manifest_check[col].max()

    print(
        f"{col:<30} "
        f"min={mn:.3f} | max={mx:.3f}"
    )

# ------------------------------------------------------------
# 6. MRI preprocessing integrity
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MRI PREPROCESSING INTEGRITY")
print("=" * 80)

print(
    "Missing P1/P99:",
    manifest_check[
        ["P1", "P99"]
    ].isna().sum().sum()
)

print(
    "Invalid series:",
    (~manifest_check["Valid"]).sum()
)

print(
    "Bad percentile ordering:",
    (manifest_check["P99"] <= manifest_check["P1"]).sum()
)

# ------------------------------------------------------------
# 7. Path sanity sample
#
# Avoid checking all 2.3 million path references.
# Just verify a representative sample.
# ------------------------------------------------------------

sample_manifest = manifest_check.sample(
    n=min(500, len(manifest_check)),
    random_state=123
)

path_cols = [
    "PreviousPath",
    "CentrePath",
    "NextPath",
]

print("\nSample path checks:")

for col in path_cols:

    missing_paths = (
        ~sample_manifest[col]
        .map(lambda x: Path(x).exists())
    ).sum()

    print(
        f"{col:<20} missing={missing_paths}"
    )

# ------------------------------------------------------------
# 8. Save compact study summary for GPU notebook
# ------------------------------------------------------------

STUDY_SUMMARY_PATH = (
    CPU_OUT /
    "study_supervision_summary.csv"
)

study_summary.to_csv(
    STUDY_SUMMARY_PATH,
    index=False
)

print("\nSaved study summary:")
print(STUDY_SUMMARY_PATH)

# ------------------------------------------------------------
# 9. Critical assertions
# ------------------------------------------------------------

assert len(manifest_check) == 770334

assert (
    manifest_check["StudyInstanceUID"].nunique()
    == 4407
)

assert (
    manifest_check["SeriesInstanceUID"].nunique()
    == 24371
)

assert len(percentiles_check) == 24371

assert (
    study_summary["SupervisionSource"]
    .value_counts()
    .get("Gold", 0)
    == 58
)

assert (
    study_summary["SupervisionSource"]
    .value_counts()
    .get("Pseudo", 0)
    == 4349
)

assert (
    manifest_check[target_cols]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    manifest_check[
        ["P1", "P99"]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    manifest_check["P99"]
    > manifest_check["P1"]
).all()

assert (
    manifest_check["Valid"]
).all()

assert len(weights_check) == 12

assert config_check["num_targets"] == 12

# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("✓ FINAL CPU PREPARATION PASSED")
print("✓ 4,407 studies")
print("✓ 24,371 MRI series")
print("✓ 770,334 supervised triplets")
print("✓ 58 gold studies")
print("✓ 4,349 pseudo studies")
print("✓ Targets and normalization validated")
print("✓ Training config validated")
print("✓ CPU notebook ready to Save Version")
print("✓ NEXT NOTEBOOK = GPU TRAINING")
print("=" * 80)